# Healthcare Operations & Patient Performance Analytics

## Data Preparation, Quality Audit & Transformation

### Project Objective

This notebook prepares healthcare data for Tableau analysis by:

- Profiling the raw datasets.
- Auditing data quality and integrity.
- Cleaning and standardizing data.
- Creating analytical fields for BI reporting.
- Validating the transformed datasets.
- Calculating KPI control totals.
- Exporting a final Tableau-ready Excel workbook.

### Data Governance

- The original raw workbook remains unchanged.
- A working copy is uploaded to Colab.
- All transformations are performed on the working copy.
- The final transformed workbook is exported separately.

### Workflow

Raw Workbook → Profiling → Quality Audit → Cleaning → Standardization → Transformation → Validation → Tableau

## 02. Environment Setup

Import the Python libraries required for data preparation, quality auditing, transformation, validation, and Excel processing.

In [1]:
# Import standard libraries
import io
import os
import warnings
from pathlib import Path

# Import data analysis libraries
import pandas as pd
import numpy as np

# Import Google Colab file upload/download utilities
from google.colab import files

# Suppress unnecessary warnings
warnings.filterwarnings("ignore")

# Configure pandas display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

# Reproducibility
RANDOM_STATE = 42

# Display environment information
print("Environment setup completed.")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Environment setup completed.
Pandas version: 2.2.3
NumPy version: 2.1.3


## 03. Import Raw Workbook

Upload the working copy of the raw healthcare workbook into Google Colab.

The original backup stored on the Desktop remains unchanged.

In [2]:
# Open Colab's native file upload dialog
uploaded = files.upload()

# Validate file selection
if not uploaded:
    raise FileNotFoundError("No file was uploaded.")

# Get the uploaded filename
RAW_FILE = next(iter(uploaded))

# Load the uploaded workbook into pandas
excel = pd.ExcelFile(io.BytesIO(uploaded[RAW_FILE]))

# Store sheet names
sheet_names = excel.sheet_names

# Display import confirmation
print("Workbook imported successfully.")
print(f"File: {RAW_FILE}")
print(f"Total sheets: {len(sheet_names)}")

Saving Healthcare_Operations_Patient_Performance_Analytics.xlsx to Healthcare_Operations_Patient_Performance_Analytics.xlsx
Workbook imported successfully.
File: Healthcare_Operations_Patient_Performance_Analytics.xlsx
Total sheets: 18


## 04. Workbook & Sheet Inventory

Review the workbook structure and separate the 16 working datasets from the 2 informational sheets.

In [3]:
# Define informational sheets
INFO_SHEETS = [
    "MODEL_GUIDE",
    "DATA_DICTIONARY"
]

# Identify working datasets
WORKING_SHEETS = [
    sheet for sheet in sheet_names
    if sheet not in INFO_SHEETS
]

# Display workbook structure
print(f"Total sheets        : {len(sheet_names)}")
print(f"Working datasets    : {len(WORKING_SHEETS)}")
print(f"Informational sheets: {len(INFO_SHEETS)}")

print("\nWorking datasets:")
for i, sheet in enumerate(WORKING_SHEETS, start=1):
    print(f"{i:02d}. {sheet}")

print("\nInformational sheets:")
for sheet in INFO_SHEETS:
    print(f"- {sheet}")

Total sheets        : 18
Working datasets    : 16
Informational sheets: 2

Working datasets:
01. FACT_ENCOUNTER
02. DIM_PATIENT
03. DIM_DATE
04. DIM_FACILITY
05. DIM_DEPARTMENT
06. DIM_PHYSICIAN
07. DIM_SERVICE
08. DIM_DIAGNOSIS
09. DIM_PAYER
10. APPOINTMENTS
11. BED_UTILIZATION
12. CLAIMS
13. PATIENT_SURVEYS
14. PHARMACY
15. LAB_TRANSACTIONS
16. IMAGING

Informational sheets:
- MODEL_GUIDE
- DATA_DICTIONARY


## 05. Dataset Profiling

Load all working datasets and review their basic structure before performing any cleaning or transformation.

The profiling stage checks:

- Row count
- Column count
- Missing values
- Duplicate rows
- Data types

In [7]:
# Load all 16 working datasets into a dictionary
datasets = {}

for sheet in WORKING_SHEETS:
    datasets[sheet] = pd.read_excel(excel, sheet_name=sheet)

# Create profiling summary
profile = []

for sheet, df in datasets.items():
    profile.append({
        "Dataset": sheet,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing Values": int(df.isna().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum()),
        "Memory (MB)": round(df.memory_usage(deep=True).sum() / 1024**2, 2)
    })

# Convert results to DataFrame
profile_df = pd.DataFrame(profile)

# Display profiling summary
profile_df

,Dataset,Rows,Columns,Missing Values,Duplicate Rows,Memory (MB)
0,FACT_ENCOUNTER,75000,26,0,0,65.11
1,DIM_PATIENT,30000,10,0,0,11.65
2,DIM_DATE,1096,10,0,0,0.27
3,DIM_FACILITY,10,8,0,0,0.00
4,DIM_DEPARTMENT,15,6,0,0,0.00
5,DIM_PHYSICIAN,60,7,0,0,0.02
6,DIM_SERVICE,25,6,0,0,0.01
7,DIM_DIAGNOSIS,30,3,0,0,0.01
8,DIM_PAYER,8,5,0,0,0.00
9,APPOINTMENTS,5000,11,0,0,2.27


In [8]:
# Confirm that all expected datasets were loaded
print(f"Datasets loaded: {len(datasets)}")

# Display each dataset shape
for sheet, df in datasets.items():
    print(f"{sheet:<20} {df.shape}")

Datasets loaded: 16
FACT_ENCOUNTER       (75000, 26)
DIM_PATIENT          (30000, 10)
DIM_DATE             (1096, 10)
DIM_FACILITY         (10, 8)
DIM_DEPARTMENT       (15, 6)
DIM_PHYSICIAN        (60, 7)
DIM_SERVICE          (25, 6)
DIM_DIAGNOSIS        (30, 3)
DIM_PAYER            (8, 5)
APPOINTMENTS         (5000, 11)
BED_UTILIZATION      (10960, 7)
CLAIMS               (8000, 11)
PATIENT_SURVEYS      (6000, 11)
PHARMACY             (7000, 10)
LAB_TRANSACTIONS     (7000, 10)
IMAGING              (5000, 11)


## 06. Data Quality Audit

Evaluate the working datasets against key data-quality dimensions:

- Completeness
- Uniqueness
- Validity
- Consistency
- Referential integrity
- Business-rule compliance

No data will be modified during the audit.

In [9]:
# Create a quality audit summary
quality_audit = []

for sheet, df in datasets.items():

    quality_audit.append({
        "Dataset": sheet,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing Values": int(df.isna().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum()),
        "Missing %": round(
            df.isna().sum().sum() / df.size * 100, 2
        )
    })

# Convert results to DataFrame
quality_audit_df = pd.DataFrame(quality_audit)

# Display the audit summary
quality_audit_df

,Dataset,Rows,Columns,Missing Values,Duplicate Rows,Missing %
0,FACT_ENCOUNTER,75000,26,0,0,0.00
1,DIM_PATIENT,30000,10,0,0,0.00
2,DIM_DATE,1096,10,0,0,0.00
3,DIM_FACILITY,10,8,0,0,0.00
4,DIM_DEPARTMENT,15,6,0,0,0.00
5,DIM_PHYSICIAN,60,7,0,0,0.00
6,DIM_SERVICE,25,6,0,0,0.00
7,DIM_DIAGNOSIS,30,3,0,0,0.00
8,DIM_PAYER,8,5,0,0,0.00
9,APPOINTMENTS,5000,11,0,0,0.00


### Key Structure Audit

Inspect the actual identifier columns in each dataset before validating primary and natural keys.

This prevents incorrect assumptions about column names and ensures each dataset is validated according to its actual grain.

In [10]:
# Display the actual columns for every working dataset

for sheet, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(sheet)
    print(f"{'=' * 60}")

    for column in df.columns:
        print(f"- {column}")


FACT_ENCOUNTER
- Encounter_ID
- Patient_ID
- Date_ID
- Facility_ID
- Department_ID
- Physician_ID
- Service_ID
- Diagnosis_ID
- Payer_ID
- Encounter_Type
- Admission_Source
- Admission_Date
- Discharge_Date
- Wait_Time_Minutes
- Treatment_Time_Minutes
- Length_of_Stay_Days
- Service_Revenue_SAR
- Service_Cost_SAR
- Gross_Profit_SAR
- Patient_Satisfaction_Score
- Readmission_Flag
- Outcome
- Discharge_Disposition
- Appointment_Flag
- Cancellation_Flag
- Profit_Margin_Pct

DIM_PATIENT
- Patient_ID
- Gender
- Age
- Age_Group
- City
- Region
- Nationality_Group
- Registration_Date
- Chronic_Condition_Flag
- Date_of_Birth

DIM_DATE
- Date_ID
- Date
- Year
- Quarter
- Month_Number
- Month_Name
- Month_Year
- Week_Number
- Day_Name
- Is_Weekend

DIM_FACILITY
- Facility_ID
- Facility_Name
- City
- Region
- Facility_Type
- Ownership
- Bed_Capacity
- Opening_Year

DIM_DEPARTMENT
- Department_ID
- Department_Name
- Department_Group
- Department_Bed_Capacity
- Target_Utilization
- Target_Wait_Min

In [11]:
# Identify ID-like columns in each dataset

for sheet, df in datasets.items():

    id_columns = [
        column for column in df.columns
        if "ID" in str(column).upper()
    ]

    print(f"\n{sheet}")
    print("-" * len(sheet))

    if id_columns:
        for column in id_columns:
            print(f"{column}: {df[column].nunique(dropna=True):,} unique values")
    else:
        print("No ID-like columns found.")


FACT_ENCOUNTER
--------------
Encounter_ID: 75,000 unique values
Patient_ID: 27,532 unique values
Date_ID: 1,096 unique values
Facility_ID: 10 unique values
Department_ID: 15 unique values
Physician_ID: 60 unique values
Service_ID: 24 unique values
Diagnosis_ID: 30 unique values
Payer_ID: 8 unique values

DIM_PATIENT
-----------
Patient_ID: 30,000 unique values

DIM_DATE
--------
Date_ID: 1,096 unique values

DIM_FACILITY
------------
Facility_ID: 10 unique values

DIM_DEPARTMENT
--------------
Department_ID: 15 unique values

DIM_PHYSICIAN
-------------
Physician_ID: 60 unique values
Department_ID: 15 unique values

DIM_SERVICE
-----------
Service_ID: 25 unique values

DIM_DIAGNOSIS
-------------
Diagnosis_ID: 30 unique values

DIM_PAYER
---------
Payer_ID: 8 unique values

APPOINTMENTS
------------
Appointment_ID: 5,000 unique values
Patient_ID: 4,588 unique values
Facility_ID: 10 unique values
Department_ID: 15 unique values
Physician_ID: 60 unique values

BED_UTILIZATION
---------

### Primary Key & Natural Key Audit

Validate that each dataset has complete and unique primary or natural keys based on its actual structure and grain.

In [12]:
# Define the actual primary keys from the workbook schema
primary_keys = {
    "FACT_ENCOUNTER": "Encounter_ID",
    "DIM_PATIENT": "Patient_ID",
    "DIM_DATE": "Date_ID",
    "DIM_FACILITY": "Facility_ID",
    "DIM_DEPARTMENT": "Department_ID",
    "DIM_PHYSICIAN": "Physician_ID",
    "DIM_SERVICE": "Service_ID",
    "DIM_DIAGNOSIS": "Diagnosis_ID",
    "DIM_PAYER": "Payer_ID",
    "APPOINTMENTS": "Appointment_ID",
    "BED_UTILIZATION": "Bed_Record_ID",
    "CLAIMS": "Claim_ID",
    "PATIENT_SURVEYS": "Survey_ID",
    "PHARMACY": "Pharmacy_Transaction_ID",
    "LAB_TRANSACTIONS": "Lab_Test_ID",
    "IMAGING": "Imaging_ID"
}

# Store audit results
key_audit = []

# Validate each primary key
for sheet, key in primary_keys.items():

    df = datasets[sheet]

    # Count missing key values
    missing_keys = df[key].isna().sum()

    # Count duplicated key values
    duplicate_keys = df[key].duplicated().sum()

    # Determine audit status
    status = "PASS" if missing_keys == 0 and duplicate_keys == 0 else "REVIEW"

    # Store results
    key_audit.append({
        "Dataset": sheet,
        "Primary Key": key,
        "Missing Keys": int(missing_keys),
        "Duplicate Keys": int(duplicate_keys),
        "Status": status
    })

# Convert results to DataFrame
key_audit_df = pd.DataFrame(key_audit)

# Display audit results
key_audit_df

,Dataset,Primary Key,Missing Keys,Duplicate Keys,Status
0,FACT_ENCOUNTER,Encounter_ID,0,0,PASS
1,DIM_PATIENT,Patient_ID,0,0,PASS
2,DIM_DATE,Date_ID,0,0,PASS
3,DIM_FACILITY,Facility_ID,0,0,PASS
4,DIM_DEPARTMENT,Department_ID,0,0,PASS
5,DIM_PHYSICIAN,Physician_ID,0,0,PASS
6,DIM_SERVICE,Service_ID,0,0,PASS
7,DIM_DIAGNOSIS,Diagnosis_ID,0,0,PASS
8,DIM_PAYER,Payer_ID,0,0,PASS
9,APPOINTMENTS,Appointment_ID,0,0,PASS


In [13]:
# Summarize primary-key audit results
print(f"Datasets audited: {len(key_audit_df)}")
print(f"Passed: {(key_audit_df['Status'] == 'PASS').sum()}")
print(f"Requires review: {(key_audit_df['Status'] == 'REVIEW').sum()}")

Datasets audited: 16
Passed: 16
Requires review: 0


### Data Validity Audit

Validate important fields using the actual workbook schema.

The audit checks value ranges, financial logic, dates, and operational constraints without modifying the data.

In [14]:
# Display columns for datasets used in the validity audit

audit_datasets = [
    "FACT_ENCOUNTER",
    "DIM_PATIENT",
    "BED_UTILIZATION",
    "CLAIMS",
    "PATIENT_SURVEYS",
    "PHARMACY",
    "LAB_TRANSACTIONS",
    "IMAGING"
]

for sheet in audit_datasets:
    print(f"\n{sheet}")
    print("-" * len(sheet))
    print(list(datasets[sheet].columns))


FACT_ENCOUNTER
--------------
['Encounter_ID', 'Patient_ID', 'Date_ID', 'Facility_ID', 'Department_ID', 'Physician_ID', 'Service_ID', 'Diagnosis_ID', 'Payer_ID', 'Encounter_Type', 'Admission_Source', 'Admission_Date', 'Discharge_Date', 'Wait_Time_Minutes', 'Treatment_Time_Minutes', 'Length_of_Stay_Days', 'Service_Revenue_SAR', 'Service_Cost_SAR', 'Gross_Profit_SAR', 'Patient_Satisfaction_Score', 'Readmission_Flag', 'Outcome', 'Discharge_Disposition', 'Appointment_Flag', 'Cancellation_Flag', 'Profit_Margin_Pct']

DIM_PATIENT
-----------
['Patient_ID', 'Gender', 'Age', 'Age_Group', 'City', 'Region', 'Nationality_Group', 'Registration_Date', 'Chronic_Condition_Flag', 'Date_of_Birth']

BED_UTILIZATION
---------------
['Date', 'Facility_ID', 'Bed_Capacity', 'Occupied_Beds', 'Available_Beds', 'Bed_Utilization_Pct', 'Bed_Record_ID']

CLAIMS
------
['Claim_ID', 'Encounter_ID', 'Patient_ID', 'Payer_ID', 'Claim_Date', 'Submitted_Amount_SAR', 'Approved_Amount_SAR', 'Paid_Amount_SAR', 'Claim_Stat

In [15]:
# Store validity audit results
validity_audit = []


# Helper function to record audit results
def add_check(dataset, check, invalid_count):
    validity_audit.append({
        "Dataset": dataset,
        "Check": check,
        "Invalid Records": int(invalid_count),
        "Status": "PASS" if invalid_count == 0 else "REVIEW"
    })


# =========================================================
# DIM_PATIENT
# =========================================================

df = datasets["DIM_PATIENT"]

add_check(
    "DIM_PATIENT",
    "Age between 0 and 120",
    ((df["Age"] < 0) | (df["Age"] > 120)).sum()
)


# =========================================================
# FACT_ENCOUNTER
# =========================================================

df = datasets["FACT_ENCOUNTER"]

add_check(
    "FACT_ENCOUNTER",
    "Revenue >= 0",
    (df["Service_Revenue_SAR"] < 0).sum()
)

add_check(
    "FACT_ENCOUNTER",
    "Cost >= 0",
    (df["Service_Cost_SAR"] < 0).sum()
)

add_check(
    "FACT_ENCOUNTER",
    "Wait time >= 0",
    (df["Wait_Time_Minutes"] < 0).sum()
)

add_check(
    "FACT_ENCOUNTER",
    "Treatment time >= 0",
    (df["Treatment_Time_Minutes"] < 0).sum()
)

add_check(
    "FACT_ENCOUNTER",
    "Length of stay >= 0",
    (df["Length_of_Stay_Days"] < 0).sum()
)

add_check(
    "FACT_ENCOUNTER",
    "Satisfaction between 1 and 5",
    (
        (df["Patient_Satisfaction_Score"] < 1) |
        (df["Patient_Satisfaction_Score"] > 5)
    ).sum()
)

# Validate gross profit calculation
gross_profit_error = (
    ~np.isclose(
        df["Gross_Profit_SAR"],
        df["Service_Revenue_SAR"] - df["Service_Cost_SAR"],
        atol=0.01
    )
).sum()

add_check(
    "FACT_ENCOUNTER",
    "Gross profit = Revenue - Cost",
    gross_profit_error
)

# Validate admission and discharge dates
admission_date = pd.to_datetime(df["Admission_Date"])
discharge_date = pd.to_datetime(df["Discharge_Date"])

add_check(
    "FACT_ENCOUNTER",
    "Discharge date >= admission date",
    (discharge_date < admission_date).sum()
)


# =========================================================
# BED_UTILIZATION
# =========================================================

df = datasets["BED_UTILIZATION"]

add_check(
    "BED_UTILIZATION",
    "Utilization between 0 and 100%",
    (
        (df["Bed_Utilization_Pct"] < 0) |
        (df["Bed_Utilization_Pct"] > 100)
    ).sum()
)

add_check(
    "BED_UTILIZATION",
    "Occupied beds <= capacity",
    (df["Occupied_Beds"] > df["Bed_Capacity"]).sum()
)

add_check(
    "BED_UTILIZATION",
    "Available beds >= 0",
    (df["Available_Beds"] < 0).sum()
)


# =========================================================
# CLAIMS
# =========================================================

df = datasets["CLAIMS"]

add_check(
    "CLAIMS",
    "Submitted amount >= 0",
    (df["Submitted_Amount_SAR"] < 0).sum()
)

add_check(
    "CLAIMS",
    "Approved amount >= 0",
    (df["Approved_Amount_SAR"] < 0).sum()
)

add_check(
    "CLAIMS",
    "Paid amount >= 0",
    (df["Paid_Amount_SAR"] < 0).sum()
)

add_check(
    "CLAIMS",
    "Approved amount <= submitted amount",
    (df["Approved_Amount_SAR"] > df["Submitted_Amount_SAR"]).sum()
)

add_check(
    "CLAIMS",
    "Paid amount <= approved amount",
    (df["Paid_Amount_SAR"] > df["Approved_Amount_SAR"]).sum()
)

add_check(
    "CLAIMS",
    "Outstanding amount >= 0",
    (df["Outstanding_Amount_SAR"] < 0).sum()
)

add_check(
    "CLAIMS",
    "Days to payment >= 0",
    (df["Days_to_Payment"] < 0).sum()
)


# =========================================================
# PATIENT_SURVEYS
# =========================================================

df = datasets["PATIENT_SURVEYS"]

survey_columns = [
    "Overall_Satisfaction",
    "Staff_Rating",
    "Facility_Rating",
    "Wait_Time_Rating",
    "Cleanliness_Rating"
]

for column in survey_columns:

    invalid = (
        (df[column] < 1) |
        (df[column] > 5)
    ).sum()

    add_check(
        "PATIENT_SURVEYS",
        f"{column} between 1 and 5",
        invalid
    )


# =========================================================
# PHARMACY
# =========================================================

df = datasets["PHARMACY"]

add_check(
    "PHARMACY",
    "Quantity >= 0",
    (df["Quantity"] < 0).sum()
)

add_check(
    "PHARMACY",
    "Revenue >= 0",
    (df["Revenue_SAR"] < 0).sum()
)

add_check(
    "PHARMACY",
    "Cost >= 0",
    (df["Cost_SAR"] < 0).sum()
)

add_check(
    "PHARMACY",
    "Gross profit = Revenue - Cost",
    (
        ~np.isclose(
            df["Gross_Profit_SAR"],
            df["Revenue_SAR"] - df["Cost_SAR"],
            atol=0.01
        )
    ).sum()
)


# =========================================================
# LAB_TRANSACTIONS
# =========================================================

df = datasets["LAB_TRANSACTIONS"]

add_check(
    "LAB_TRANSACTIONS",
    "Processing time >= 0",
    (df["Processing_Time_Hours"] < 0).sum()
)

add_check(
    "LAB_TRANSACTIONS",
    "Revenue >= 0",
    (df["Revenue_SAR"] < 0).sum()
)

add_check(
    "LAB_TRANSACTIONS",
    "Cost >= 0",
    (df["Cost_SAR"] < 0).sum()
)


# =========================================================
# IMAGING
# =========================================================

df = datasets["IMAGING"]

add_check(
    "IMAGING",
    "Waiting time >= 0",
    (df["Waiting_Time_Minutes"] < 0).sum()
)

add_check(
    "IMAGING",
    "Report time >= 0",
    (df["Report_Time_Hours"] < 0).sum()
)

add_check(
    "IMAGING",
    "Revenue >= 0",
    (df["Revenue_SAR"] < 0).sum()
)

add_check(
    "IMAGING",
    "Cost >= 0",
    (df["Cost_SAR"] < 0).sum()
)


# Convert results to DataFrame
validity_audit_df = pd.DataFrame(validity_audit)

# Display audit results
validity_audit_df

,Dataset,Check,Invalid Records,Status
0,DIM_PATIENT,Age between 0 and 120,0,PASS
1,FACT_ENCOUNTER,Revenue >= 0,0,PASS
2,FACT_ENCOUNTER,Cost >= 0,0,PASS
3,FACT_ENCOUNTER,Wait time >= 0,0,PASS
4,FACT_ENCOUNTER,Treatment time >= 0,0,PASS
5,FACT_ENCOUNTER,Length of stay >= 0,0,PASS
6,FACT_ENCOUNTER,Satisfaction between 1 and 5,0,PASS
7,FACT_ENCOUNTER,Gross profit = Revenue - Cost,0,PASS
8,FACT_ENCOUNTER,Discharge date >= admission date,0,PASS
9,BED_UTILIZATION,Utilization between 0 and 100%,0,PASS


### Claims Exception Investigation

Investigate the claim records that failed financial consistency checks.

The objective is to understand the issue before applying any cleaning rule.

In [16]:
# Select the claims dataset
claims = datasets["CLAIMS"].copy()

# Identify financial exceptions
paid_above_approved = claims[
    claims["Paid_Amount_SAR"] > claims["Approved_Amount_SAR"]
].copy()

negative_outstanding = claims[
    claims["Outstanding_Amount_SAR"] < 0
].copy()

# Display exception counts
print(f"Paid > Approved exceptions : {len(paid_above_approved):,}")
print(f"Negative Outstanding       : {len(negative_outstanding):,}")

Paid > Approved exceptions : 2,624
Negative Outstanding       : 160


In [17]:
# Display the key financial fields for the exceptions

claims[
    (claims["Paid_Amount_SAR"] > claims["Approved_Amount_SAR"]) |
    (claims["Outstanding_Amount_SAR"] < 0)
][[
    "Claim_ID",
    "Claim_Status",
    "Submitted_Amount_SAR",
    "Approved_Amount_SAR",
    "Paid_Amount_SAR",
    "Outstanding_Amount_SAR",
    "Days_to_Payment"
]].head(20)

,Claim_ID,Claim_Status,Submitted_Amount_SAR,Approved_Amount_SAR,Paid_Amount_SAR,Outstanding_Amount_SAR,Days_to_Payment
7,CLM000008,Paid,172.77,145.38,155.66,17.11,42
10,CLM000011,Paid,"1,232.13","1,167.69","1,219.77",12.36,73
11,CLM000012,Paid,940.99,913.10,952.48,-11.49,47
12,CLM000013,Paid,610.62,502.01,544.95,65.67,59
13,CLM000014,Paid,"1,538.21","1,381.39","1,388.74",149.47,58
14,CLM000015,Paid,"1,117.89","1,001.89","1,008.57",109.32,12
19,CLM000020,Paid,"1,235.61","1,120.08","1,222.92",12.69,47
22,CLM000023,Paid,"1,367.66","1,177.33","1,297.60",70.06,38
24,CLM000025,Paid,822.19,696.58,783.07,39.12,18
25,CLM000026,Paid,160.69,137.90,147.12,13.57,10


In [18]:
# Calculate financial differences to understand the pattern

claims["Paid_vs_Approved"] = (
    claims["Paid_Amount_SAR"] -
    claims["Approved_Amount_SAR"]
)

claims["Expected_Outstanding"] = (
    claims["Submitted_Amount_SAR"] -
    claims["Paid_Amount_SAR"]
)

# Summarize the exceptions
print("Paid above approved:")
print(
    claims.loc[
        claims["Paid_vs_Approved"] > 0,
        "Paid_vs_Approved"
    ].describe()
)

print("\nNegative outstanding:")
print(
    claims.loc[
        claims["Outstanding_Amount_SAR"] < 0,
        "Outstanding_Amount_SAR"
    ].describe()
)

Paid above approved:
count   2,624.00
mean       63.79
std        92.14
min         0.05
25%        13.72
50%        35.59
75%        76.78
max     1,257.34
Name: Paid_vs_Approved, dtype: float64

Negative outstanding:
count    160.00
mean     -19.81
std       21.45
min     -125.35
25%      -26.82
50%      -13.34
75%       -6.24
max       -0.02
Name: Outstanding_Amount_SAR, dtype: float64


In [19]:
# Compare the stored outstanding amount against both possible calculations

claims["Outstanding_vs_Submitted"] = (
    claims["Outstanding_Amount_SAR"] -
    (
        claims["Submitted_Amount_SAR"] -
        claims["Paid_Amount_SAR"]
    )
)

claims["Outstanding_vs_Approved"] = (
    claims["Outstanding_Amount_SAR"] -
    (
        claims["Approved_Amount_SAR"] -
        claims["Paid_Amount_SAR"]
    )
)

print(
    "Matches Submitted - Paid:",
    np.isclose(
        claims["Outstanding_vs_Submitted"],
        0,
        atol=0.01
    ).sum()
)

print(
    "Matches Approved - Paid:",
    np.isclose(
        claims["Outstanding_vs_Approved"],
        0,
        atol=0.01
    ).sum()
)

Matches Submitted - Paid: 8000
Matches Approved - Paid: 0


### Claims Business Rule Decision

The claims audit identified inconsistencies where paid amounts exceed approved amounts and, in some cases, submitted amounts.

The audit confirms that:

Outstanding Amount = Submitted Amount - Paid Amount

Therefore, the submitted amount remains the basis for outstanding balance calculation.

The approved and paid amounts will be reviewed for logical consistency before final transformation.

In [20]:
# Calculate the financial impact of the claims exceptions

claims = datasets["CLAIMS"].copy()

# Identify records where paid amount exceeds approved amount
paid_above_approved = claims[
    claims["Paid_Amount_SAR"] > claims["Approved_Amount_SAR"]
]

# Identify records where paid amount exceeds submitted amount
paid_above_submitted = claims[
    claims["Paid_Amount_SAR"] > claims["Submitted_Amount_SAR"]
]

# Calculate exception counts and amounts
print("Claims exception impact")
print("-" * 35)

print(
    f"Paid > Approved records : "
    f"{len(paid_above_approved):,}"
)

print(
    f"Paid > Submitted records: "
    f"{len(paid_above_submitted):,}"
)

print(
    f"Paid > Submitted amount : "
    f"SAR {paid_above_submitted['Paid_Amount_SAR'].sum() - paid_above_submitted['Submitted_Amount_SAR'].sum():,.2f}"
)

Claims exception impact
-----------------------------------
Paid > Approved records : 2,624
Paid > Submitted records: 160
Paid > Submitted amount : SAR 3,169.48


In [21]:
# Check the relationship between claim status and paid amount exceptions

claims["Paid_Above_Submitted"] = (
    claims["Paid_Amount_SAR"] >
    claims["Submitted_Amount_SAR"]
)

claims["Paid_Above_Approved"] = (
    claims["Paid_Amount_SAR"] >
    claims["Approved_Amount_SAR"]
)

status_summary = (
    claims.groupby("Claim_Status")
    .agg(
        Claims=("Claim_ID", "count"),
        Paid_Above_Approved=("Paid_Above_Approved", "sum"),
        Paid_Above_Submitted=("Paid_Above_Submitted", "sum")
    )
    .reset_index()
)

status_summary

,Claim_Status,Claims,Paid_Above_Approved,Paid_Above_Submitted
0,Paid,6154,2023,116
1,Partially Paid,729,224,13
2,Rejected,462,157,12
3,Under Review,655,220,19


### Referential Integrity Audit

Validate relationships between transactional datasets and their related dimension records.

The audit identifies orphan foreign keys that could cause unmatched records in Tableau relationships.

In [22]:
# Define foreign-key relationships
relationships = {
    "FACT_ENCOUNTER": {
        "Patient_ID": "DIM_PATIENT",
        "Date_ID": "DIM_DATE",
        "Facility_ID": "DIM_FACILITY",
        "Department_ID": "DIM_DEPARTMENT",
        "Physician_ID": "DIM_PHYSICIAN",
        "Service_ID": "DIM_SERVICE",
        "Diagnosis_ID": "DIM_DIAGNOSIS",
        "Payer_ID": "DIM_PAYER"
    },

    "APPOINTMENTS": {
        "Patient_ID": "DIM_PATIENT",
        "Facility_ID": "DIM_FACILITY",
        "Department_ID": "DIM_DEPARTMENT",
        "Physician_ID": "DIM_PHYSICIAN"
    },

    "CLAIMS": {
        "Encounter_ID": "FACT_ENCOUNTER",
        "Patient_ID": "DIM_PATIENT",
        "Payer_ID": "DIM_PAYER"
    },

    "PATIENT_SURVEYS": {
        "Encounter_ID": "FACT_ENCOUNTER",
        "Patient_ID": "DIM_PATIENT"
    },

    "PHARMACY": {
        "Patient_ID": "DIM_PATIENT",
        "Encounter_ID": "FACT_ENCOUNTER"
    },

    "LAB_TRANSACTIONS": {
        "Patient_ID": "DIM_PATIENT",
        "Encounter_ID": "FACT_ENCOUNTER"
    },

    "IMAGING": {
        "Patient_ID": "DIM_PATIENT",
        "Encounter_ID": "FACT_ENCOUNTER"
    },

    "BED_UTILIZATION": {
        "Facility_ID": "DIM_FACILITY"
    }
}


# Store audit results
referential_audit = []


# Run each relationship check
for source_table, mappings in relationships.items():

    source_df = datasets[source_table]

    for source_key, target_table in mappings.items():

        target_df = datasets[target_table]

        # Get unique target keys
        target_values = set(
            target_df[source_key].dropna().unique()
        )

        # Find source values not present in target
        orphan_mask = ~source_df[source_key].isin(target_values)

        orphan_count = orphan_mask.sum()

        # Store result
        referential_audit.append({
            "Source Dataset": source_table,
            "Foreign Key": source_key,
            "Target Dataset": target_table,
            "Orphan Records": int(orphan_count),
            "Status": "PASS" if orphan_count == 0 else "REVIEW"
        })


# Convert results to DataFrame
referential_audit_df = pd.DataFrame(referential_audit)

# Display results
referential_audit_df

,Source Dataset,Foreign Key,Target Dataset,Orphan Records,Status
0,FACT_ENCOUNTER,Patient_ID,DIM_PATIENT,0,PASS
1,FACT_ENCOUNTER,Date_ID,DIM_DATE,0,PASS
2,FACT_ENCOUNTER,Facility_ID,DIM_FACILITY,0,PASS
3,FACT_ENCOUNTER,Department_ID,DIM_DEPARTMENT,0,PASS
4,FACT_ENCOUNTER,Physician_ID,DIM_PHYSICIAN,0,PASS
5,FACT_ENCOUNTER,Service_ID,DIM_SERVICE,0,PASS
6,FACT_ENCOUNTER,Diagnosis_ID,DIM_DIAGNOSIS,0,PASS
7,FACT_ENCOUNTER,Payer_ID,DIM_PAYER,0,PASS
8,APPOINTMENTS,Patient_ID,DIM_PATIENT,0,PASS
9,APPOINTMENTS,Facility_ID,DIM_FACILITY,0,PASS


In [23]:
# Summarize referential integrity results

total_relationships = len(referential_audit_df)
passed_relationships = (
    referential_audit_df["Status"] == "PASS"
).sum()
review_relationships = (
    referential_audit_df["Status"] == "REVIEW"
).sum()

print(f"Relationships audited : {total_relationships}")
print(f"Passed                : {passed_relationships}")
print(f"Requires review       : {review_relationships}")

Relationships audited : 24
Passed                : 24
Requires review       : 0


### Date & Temporal Validation

Validate date fields for correct formatting, project-period consistency, and logical sequencing.

No data will be modified during this audit.

In [24]:
# Define the project date range
PROJECT_START = pd.Timestamp("2023-01-01")
PROJECT_END = pd.Timestamp("2025-12-31")

# Define date fields to audit
date_fields = {
    "FACT_ENCOUNTER": [
        "Admission_Date",
        "Discharge_Date"
    ],
    "DIM_PATIENT": [
        "Registration_Date",
        "Date_of_Birth"
    ],
    "DIM_DATE": [
        "Date"
    ],
    "APPOINTMENTS": [
        "Appointment_Date"
    ],
    "BED_UTILIZATION": [
        "Date"
    ],
    "CLAIMS": [
        "Claim_Date"
    ],
    "PATIENT_SURVEYS": [
        "Survey_Date"
    ],
    "PHARMACY": [
        "Transaction_Date"
    ],
    "LAB_TRANSACTIONS": [
        "Test_Date"
    ],
    "IMAGING": [
        "Imaging_Date"
    ]
}

# Store results
date_audit = []

# Validate each date field
for sheet, columns in date_fields.items():

    df = datasets[sheet]

    for column in columns:

        # Convert to datetime for validation only
        dates = pd.to_datetime(
            df[column],
            errors="coerce"
        )

        invalid_format = dates.isna().sum()

        outside_period = (
            ((dates < PROJECT_START) | (dates > PROJECT_END))
            & dates.notna()
        ).sum()

        date_audit.append({
            "Dataset": sheet,
            "Date Field": column,
            "Invalid Dates": int(invalid_format),
            "Outside Project Period": int(outside_period),
            "Status": (
                "PASS"
                if invalid_format == 0 and outside_period == 0
                else "REVIEW"
            )
        })

# Create result DataFrame
date_audit_df = pd.DataFrame(date_audit)

# Display results
date_audit_df

,Dataset,Date Field,Invalid Dates,Outside Project Period,Status
0,FACT_ENCOUNTER,Admission_Date,0,0,PASS
1,FACT_ENCOUNTER,Discharge_Date,0,75,REVIEW
2,DIM_PATIENT,Registration_Date,0,0,PASS
3,DIM_PATIENT,Date_of_Birth,0,30000,REVIEW
4,DIM_DATE,Date,0,0,PASS
5,APPOINTMENTS,Appointment_Date,0,0,PASS
6,BED_UTILIZATION,Date,0,0,PASS
7,CLAIMS,Claim_Date,0,0,PASS
8,PATIENT_SURVEYS,Survey_Date,0,0,PASS
9,PHARMACY,Transaction_Date,0,0,PASS


In [25]:
# Summarize date validation results

print(f"Date fields audited : {len(date_audit_df)}")
print(
    f"Passed              : "
    f"{(date_audit_df['Status'] == 'PASS').sum()}"
)
print(
    f"Requires review     : "
    f"{(date_audit_df['Status'] == 'REVIEW').sum()}"
)

Date fields audited : 12
Passed              : 10
Requires review     : 2


### Discharge Date Exception Review

Investigate encounters with discharge dates outside the project period.

Determine whether the dates represent legitimate encounter durations or data-quality exceptions before applying any transformation.

In [26]:
# Select FACT_ENCOUNTER
encounters = datasets["FACT_ENCOUNTER"].copy()

# Convert dates for analysis
encounters["Admission_Date"] = pd.to_datetime(
    encounters["Admission_Date"]
)

encounters["Discharge_Date"] = pd.to_datetime(
    encounters["Discharge_Date"]
)

# Identify discharge dates outside the project period
discharge_exceptions = encounters[
    (encounters["Discharge_Date"] < PROJECT_START) |
    (encounters["Discharge_Date"] > PROJECT_END)
].copy()

# Display exception summary
print(f"Discharge date exceptions: {len(discharge_exceptions):,}")

discharge_exceptions[
    [
        "Encounter_ID",
        "Admission_Date",
        "Discharge_Date",
        "Encounter_Type",
        "Length_of_Stay_Days",
        "Outcome"
    ]
].head(20)

Discharge date exceptions: 75


,Encounter_ID,Admission_Date,Discharge_Date,Encounter_Type,Length_of_Stay_Days,Outcome
90,ENC000091,2025-12-29,2026-01-05 14:24:00,Inpatient,7.60,Recovered
441,ENC000442,2025-12-31,2026-01-01 00:00:00,Inpatient,1.00,Improved
1411,ENC001412,2025-12-31,2026-01-01 12:00:00,Inpatient,1.50,Referred
5454,ENC005455,2025-12-31,2026-01-05 04:48:00,Inpatient,5.20,Improved
5579,ENC005580,2025-12-30,2026-01-07 19:12:00,Inpatient,8.80,Referred
7535,ENC007536,2025-12-24,2025-12-31 12:00:00,Inpatient,7.50,Recovered
7841,ENC007842,2025-12-30,2025-12-31 09:36:00,Inpatient,1.40,Improved
9761,ENC009762,2025-12-28,2026-01-02 21:36:00,Inpatient,5.90,Recovered
10904,ENC010905,2025-12-31,2026-01-10 00:00:00,Inpatient,10.00,Complication
11724,ENC011725,2025-12-24,2025-12-31 12:00:00,Inpatient,7.50,Recovered


In [27]:
# Review the earliest and latest exception dates

print("Exception date range:")
print(
    "Earliest discharge:",
    discharge_exceptions["Discharge_Date"].min()
)

print(
    "Latest discharge:",
    discharge_exceptions["Discharge_Date"].max()
)

print("\nAdmission date range:")
print(
    "Earliest admission:",
    discharge_exceptions["Admission_Date"].min()
)

print(
    "Latest admission:",
    discharge_exceptions["Admission_Date"].max()
)

Exception date range:
Earliest discharge: 2025-12-31 02:24:00
Latest discharge: 2026-01-14 07:12:00

Admission date range:
Earliest admission: 2025-12-22 00:00:00
Latest admission: 2025-12-31 00:00:00


In [28]:
# Calculate days between admission and discharge
discharge_exceptions["LOS_Calculated"] = (
    discharge_exceptions["Discharge_Date"] -
    discharge_exceptions["Admission_Date"]
).dt.total_seconds() / 86400

# Compare calculated LOS with stored LOS
discharge_exceptions["LOS_Difference"] = (
    discharge_exceptions["LOS_Calculated"] -
    discharge_exceptions["Length_of_Stay_Days"]
)

print(
    "Average calculated LOS:",
    discharge_exceptions["LOS_Calculated"].mean()
)

print(
    "Maximum calculated LOS:",
    discharge_exceptions["LOS_Calculated"].max()
)

print(
    "LOS calculation mismatches:",
    (~np.isclose(
        discharge_exceptions["LOS_Calculated"],
        discharge_exceptions["Length_of_Stay_Days"],
        atol=0.01
    )).sum()
)

Average calculated LOS: 5.553333333333334
Maximum calculated LOS: 16.3
LOS calculation mismatches: 0


## 07 - Data Type Standardization

Standardize column data types across all working datasets to ensure consistent joins, calculations, filtering, and Tableau compatibility.

Key rules:
- Identifier fields are stored as strings.
- Date fields are converted to datetime.
- Numeric measures are converted to numeric types.
- Categorical/text fields are stored consistently as strings.
- Existing values are preserved; this step only standardizes representation.

In [29]:
# Define columns by data type
ID_COLUMNS = {
    "FACT_ENCOUNTER": [
        "Encounter_ID", "Patient_ID", "Date_ID", "Facility_ID",
        "Department_ID", "Physician_ID", "Service_ID",
        "Diagnosis_ID", "Payer_ID"
    ],
    "DIM_PATIENT": ["Patient_ID"],
    "DIM_DATE": ["Date_ID"],
    "DIM_FACILITY": ["Facility_ID"],
    "DIM_DEPARTMENT": ["Department_ID"],
    "DIM_PHYSICIAN": ["Physician_ID", "Department_ID"],
    "DIM_SERVICE": ["Service_ID"],
    "DIM_DIAGNOSIS": ["Diagnosis_ID"],
    "DIM_PAYER": ["Payer_ID"],
    "APPOINTMENTS": [
        "Appointment_ID", "Patient_ID", "Facility_ID",
        "Department_ID", "Physician_ID"
    ],
    "BED_UTILIZATION": ["Bed_Record_ID", "Facility_ID"],
    "CLAIMS": ["Claim_ID", "Encounter_ID", "Patient_ID", "Payer_ID"],
    "PATIENT_SURVEYS": ["Survey_ID", "Encounter_ID", "Patient_ID"],
    "PHARMACY": [
        "Pharmacy_Transaction_ID", "Patient_ID", "Encounter_ID"
    ],
    "LAB_TRANSACTIONS": ["Lab_Test_ID", "Encounter_ID", "Patient_ID"],
    "IMAGING": ["Imaging_ID", "Encounter_ID", "Patient_ID"]
}

DATE_COLUMNS = {
    "FACT_ENCOUNTER": [
        "Admission_Date", "Discharge_Date"
    ],
    "DIM_PATIENT": [
        "Registration_Date", "Date_of_Birth"
    ],
    "DIM_DATE": ["Date"],
    "APPOINTMENTS": ["Appointment_Date"],
    "BED_UTILIZATION": ["Date"],
    "CLAIMS": ["Claim_Date"],
    "PATIENT_SURVEYS": ["Survey_Date"],
    "PHARMACY": ["Transaction_Date"],
    "LAB_TRANSACTIONS": ["Test_Date"],
    "IMAGING": ["Imaging_Date"]
}

# Standardize ID columns
for sheet, columns in ID_COLUMNS.items():
    df = datasets[sheet]

    for column in columns:
        if column in df.columns:
            df[column] = df[column].astype("string")


# Standardize date columns
for sheet, columns in DATE_COLUMNS.items():
    df = datasets[sheet]

    for column in columns:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column], errors="coerce")


# Standardize numeric columns automatically
for sheet, df in datasets.items():

    for column in df.columns:

        # Skip identifiers and dates
        if column in ID_COLUMNS.get(sheet, []):
            continue

        if column in DATE_COLUMNS.get(sheet, []):
            continue

        # Convert numeric-looking columns to numeric
        if pd.api.types.is_numeric_dtype(df[column]):
            df[column] = pd.to_numeric(df[column], errors="coerce")


print("Data type standardization completed successfully.")

Data type standardization completed successfully.


In [30]:
# Validate key data types
dtype_check = []

for sheet in WORKING_SHEETS:
    df = datasets[sheet]

    for column in ID_COLUMNS.get(sheet, []):
        if column in df.columns:
            dtype_check.append({
                "Dataset": sheet,
                "Column": column,
                "Expected": "string",
                "Actual": str(df[column].dtype),
                "Passed": str(df[column].dtype) == "string"
            })

    for column in DATE_COLUMNS.get(sheet, []):
        if column in df.columns:
            dtype_check.append({
                "Dataset": sheet,
                "Column": column,
                "Expected": "datetime",
                "Actual": str(df[column].dtype),
                "Passed": pd.api.types.is_datetime64_any_dtype(df[column])
            })

dtype_check_df = pd.DataFrame(dtype_check)

print(f"Columns checked: {len(dtype_check_df)}")
print(f"Passed: {dtype_check_df['Passed'].sum()}")
print(f"Failed: {(~dtype_check_df['Passed']).sum()}")

display(dtype_check_df[dtype_check_df["Passed"] == False])

Columns checked: 53
Passed: 53
Failed: 0


,Dataset,Column,Expected,Actual,Passed


## 08 - Claims Financial Anomaly Handling

Standardize the claims dataset while preserving the original financial values.

The transformation will:
- Flag claims where Paid Amount exceeds Approved Amount.
- Flag claims where Paid Amount exceeds Submitted Amount.
- Recalculate Outstanding Amount from Submitted Amount minus Paid Amount.
- Preserve all original submitted, approved, and paid amounts for auditability.

No financial amount will be capped or silently corrected.

In [31]:
# Select the claims dataset
claims = datasets["CLAIMS"].copy()

# Create anomaly flags
claims["Paid_Above_Approved_Flag"] = (
    claims["Paid_Amount_SAR"] > claims["Approved_Amount_SAR"]
).astype("int8")

claims["Paid_Above_Submitted_Flag"] = (
    claims["Paid_Amount_SAR"] > claims["Submitted_Amount_SAR"]
).astype("int8")

# Standardize outstanding amount using the validated source relationship
claims["Outstanding_Amount_SAR"] = (
    claims["Submitted_Amount_SAR"] - claims["Paid_Amount_SAR"]
).round(2)

# Save transformed claims dataset back
datasets["CLAIMS"] = claims

print("Claims financial anomaly handling completed successfully.")

Claims financial anomaly handling completed successfully.


In [32]:
# Validate claims anomaly flags and outstanding amount

claims = datasets["CLAIMS"]

paid_above_approved = (
    claims["Paid_Above_Approved_Flag"].sum()
)

paid_above_submitted = (
    claims["Paid_Above_Submitted_Flag"].sum()
)

outstanding_check = (
    claims["Outstanding_Amount_SAR"]
    - (claims["Submitted_Amount_SAR"] - claims["Paid_Amount_SAR"])
).abs().max()

print(f"Paid > Approved flags: {paid_above_approved:,}")
print(f"Paid > Submitted flags: {paid_above_submitted:,}")
print(f"Maximum outstanding calculation difference: {outstanding_check:.2f}")

print("\nValidation:")
print("Expected Paid > Approved: 2,624")
print("Expected Paid > Submitted: 160")
print("Expected calculation difference: 0.00")

Paid > Approved flags: 2,624
Paid > Submitted flags: 160
Maximum outstanding calculation difference: 0.00

Validation:
Expected Paid > Approved: 2,624
Expected Paid > Submitted: 160
Expected calculation difference: 0.00


### 08.X - Business Rule Validation

Validate key business relationships across the healthcare datasets to identify logically inconsistent records before transformation.

The checks focus on financial, operational, clinical, and transactional relationships while preserving the original source data.

In [33]:
# Business rule validation checks

validation_results = []

# ---------------------------------------------------------
# FACT_ENCOUNTER
# ---------------------------------------------------------
df = datasets["FACT_ENCOUNTER"]

validation_results.extend([
    {
        "Dataset": "FACT_ENCOUNTER",
        "Check": "Gross Profit = Revenue - Cost",
        "Exceptions": (
            (df["Gross_Profit_SAR"] -
             (df["Service_Revenue_SAR"] - df["Service_Cost_SAR"])).abs() > 0.01
        ).sum()
    },
    {
        "Dataset": "FACT_ENCOUNTER",
        "Check": "Profit Margin matches Gross Profit / Revenue",
        "Exceptions": (
            (
                df["Service_Revenue_SAR"] > 0
            ) &
            (
                (
                    df["Profit_Margin_Pct"] -
                    (
                        df["Gross_Profit_SAR"] /
                        df["Service_Revenue_SAR"] * 100
                    )
                ).abs() > 0.01
            )
        ).sum()
    },
    {
        "Dataset": "FACT_ENCOUNTER",
        "Check": "Discharge Date >= Admission Date",
        "Exceptions": (
            df["Discharge_Date"] < df["Admission_Date"]
        ).sum()
    }
])

# ---------------------------------------------------------
# BED_UTILIZATION
# ---------------------------------------------------------
df = datasets["BED_UTILIZATION"]

validation_results.extend([
    {
        "Dataset": "BED_UTILIZATION",
        "Check": "Occupied + Available = Capacity",
        "Exceptions": (
            df["Occupied_Beds"] +
            df["Available_Beds"] !=
            df["Bed_Capacity"]
        ).sum()
    },
    {
        "Dataset": "BED_UTILIZATION",
        "Check": "Utilization matches Occupied / Capacity",
        "Exceptions": (
            (
                df["Bed_Utilization_Pct"] -
                (
                    df["Occupied_Beds"] /
                    df["Bed_Capacity"] * 100
                )
            ).abs() > 0.01
        ).sum()
    }
])

# ---------------------------------------------------------
# CLAIMS
# ---------------------------------------------------------
df = datasets["CLAIMS"]

validation_results.extend([
    {
        "Dataset": "CLAIMS",
        "Check": "Approved Amount <= Submitted Amount",
        "Exceptions": (
            df["Approved_Amount_SAR"] >
            df["Submitted_Amount_SAR"]
        ).sum()
    },
    {
        "Dataset": "CLAIMS",
        "Check": "Outstanding = Submitted - Paid",
        "Exceptions": (
            (
                df["Outstanding_Amount_SAR"] -
                (
                    df["Submitted_Amount_SAR"] -
                    df["Paid_Amount_SAR"]
                )
            ).abs() > 0.01
        ).sum()
    }
])

# ---------------------------------------------------------
# PHARMACY
# ---------------------------------------------------------
df = datasets["PHARMACY"]

validation_results.append({
    "Dataset": "PHARMACY",
    "Check": "Gross Profit = Revenue - Cost",
    "Exceptions": (
        (
            df["Gross_Profit_SAR"] -
            (df["Revenue_SAR"] - df["Cost_SAR"])
        ).abs() > 0.01
    ).sum()
})

# ---------------------------------------------------------
# Convert results to DataFrame
# ---------------------------------------------------------
business_rule_results = pd.DataFrame(validation_results)

print("Business rule validation completed.")
display(business_rule_results)

Business rule validation completed.


,Dataset,Check,Exceptions
0,FACT_ENCOUNTER,Gross Profit = Revenue - Cost,0
1,FACT_ENCOUNTER,Profit Margin matches Gross Profit / Revenue,0
2,FACT_ENCOUNTER,Discharge Date >= Admission Date,0
3,BED_UTILIZATION,Occupied + Available = Capacity,0
4,BED_UTILIZATION,Utilization matches Occupied / Capacity,0
5,CLAIMS,Approved Amount <= Submitted Amount,0
6,CLAIMS,Outstanding = Submitted - Paid,0
7,PHARMACY,Gross Profit = Revenue - Cost,0


### 09.1 - Encounter-Level Derived Metrics

Create standardized analytical fields from the encounter fact table to support operational, financial, and patient-performance analysis in Tableau.

Derived metrics will be calculated from existing validated source fields without changing the original measures.

In [34]:
# Copy the encounter fact table before transformation
df = datasets["FACT_ENCOUNTER"].copy()

# ---------------------------------------------------------
# Financial metrics
# ---------------------------------------------------------

# Revenue generated per treatment minute
df["Revenue_Per_Treatment_Minute_SAR"] = np.where(
    df["Treatment_Time_Minutes"] > 0,
    df["Service_Revenue_SAR"] / df["Treatment_Time_Minutes"],
    0
)

# Cost incurred per treatment minute
df["Cost_Per_Treatment_Minute_SAR"] = np.where(
    df["Treatment_Time_Minutes"] > 0,
    df["Service_Cost_SAR"] / df["Treatment_Time_Minutes"],
    0
)

# ---------------------------------------------------------
# Operational metrics
# ---------------------------------------------------------

# Total time spent including waiting and treatment
df["Total_Service_Time_Minutes"] = (
    df["Wait_Time_Minutes"] +
    df["Treatment_Time_Minutes"]
)

# Wait-time target: 20 minutes or less
df["Wait_Time_Within_Target_Flag"] = (
    df["Wait_Time_Minutes"] <= 20
).astype("int8")

# Satisfaction target: 4.5 or higher
df["Satisfaction_Within_Target_Flag"] = (
    df["Patient_Satisfaction_Score"] >= 4.5
).astype("int8")

# ---------------------------------------------------------
# Standardize Yes/No flags
# ---------------------------------------------------------

# Convert Readmission Yes/No to 1/0
df["Readmission_Flag"] = (
    df["Readmission_Flag"]
    .map({"Yes": 1, "No": 0})
    .astype("int8")
)

# Convert appointment flag to 1/0 if stored as Yes/No
df["Appointment_Flag"] = (
    df["Appointment_Flag"]
    .map({"Yes": 1, "No": 0})
    .astype("int8")
)

# Convert cancellation flag to 1/0 if stored as Yes/No
df["Cancellation_Flag"] = (
    df["Cancellation_Flag"]
    .map({"Yes": 1, "No": 0})
    .astype("int8")
)

# ---------------------------------------------------------
# Encounter classification
# ---------------------------------------------------------

# Inpatient encounter flag
df["Inpatient_Flag"] = (
    df["Encounter_Type"].eq("Inpatient")
).astype("int8")

# Outpatient encounter flag
df["Outpatient_Flag"] = (
    df["Encounter_Type"].eq("Outpatient")
).astype("int8")

# ---------------------------------------------------------
# Store transformed dataset
# ---------------------------------------------------------

datasets["FACT_ENCOUNTER"] = df

print("Encounter-level feature engineering completed successfully.")

Encounter-level feature engineering completed successfully.


In [35]:
# Validate newly created encounter features

df = datasets["FACT_ENCOUNTER"]

checks = {
    "Total service time calculation": (
        df["Total_Service_Time_Minutes"]
        == df["Wait_Time_Minutes"] + df["Treatment_Time_Minutes"]
    ).all(),

    "Wait target flag": (
        df["Wait_Time_Within_Target_Flag"]
        == (df["Wait_Time_Minutes"] <= 20).astype("int8")
    ).all(),

    "Satisfaction target flag": (
        df["Satisfaction_Within_Target_Flag"]
        == (df["Patient_Satisfaction_Score"] >= 4.5).astype("int8")
    ).all(),

    "Readmission flag": (
        df["Readmission_Flag"].isin([0, 1])
    ).all(),

    "Appointment flag": (
        df["Appointment_Flag"].isin([0, 1])
    ).all(),

    "Cancellation flag": (
        df["Cancellation_Flag"].isin([0, 1])
    ).all(),

    "Inpatient flag": (
        df["Inpatient_Flag"]
        == df["Encounter_Type"].eq("Inpatient").astype("int8")
    ).all(),

    "Outpatient flag": (
        df["Outpatient_Flag"]
        == df["Encounter_Type"].eq("Outpatient").astype("int8")
    ).all()
}

print("Encounter feature validation:")

for check, passed in checks.items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

Encounter feature validation:
Total service time calculation: PASS
Wait target flag: PASS
Satisfaction target flag: PASS
Readmission flag: PASS
Appointment flag: PASS
Cancellation flag: PASS
Inpatient flag: PASS
Outpatient flag: PASS


### 09.2 - Patient-Level Derived Features

Create patient-level analytical fields to support demographic segmentation and chronic-condition analysis.

The transformation adds standardized patient segments while preserving the original demographic and clinical attributes.

In [36]:
# Copy the patient dimension before transformation
df = datasets["DIM_PATIENT"].copy()

# ---------------------------------------------------------
# Age segmentation
# ---------------------------------------------------------

# Standardize age groups for consistent Tableau analysis
df["Age_Group_Standard"] = pd.cut(
    df["Age"],
    bins=[-1, 17, 29, 44, 59, 74, 120],
    labels=[
        "0-17",
        "18-29",
        "30-44",
        "45-59",
        "60-74",
        "75+"
    ]
)

# ---------------------------------------------------------
# Chronic condition classification
# ---------------------------------------------------------

# Convert Yes/No chronic-condition flag to 1/0
df["Chronic_Condition_Flag"] = (
    df["Chronic_Condition_Flag"]
    .map({"Yes": 1, "No": 0})
    .astype("int8")
)

# ---------------------------------------------------------
# Senior patient flag
# ---------------------------------------------------------

df["Senior_Patient_Flag"] = (
    df["Age"] >= 60
).astype("int8")

# ---------------------------------------------------------
# Working-age patient flag
# ---------------------------------------------------------

df["Working_Age_Flag"] = (
    df["Age"].between(18, 59)
).astype("int8")

# ---------------------------------------------------------
# Store transformed dataset
# ---------------------------------------------------------

datasets["DIM_PATIENT"] = df

print("Patient-level feature engineering completed successfully.")

Patient-level feature engineering completed successfully.


In [37]:
# Validate patient-level derived features

df = datasets["DIM_PATIENT"]

checks = {
    "Age group populated": (
        df["Age_Group_Standard"].notna()
    ).all(),

    "Chronic condition flag": (
        df["Chronic_Condition_Flag"].isin([0, 1])
    ).all(),

    "Senior patient flag": (
        df["Senior_Patient_Flag"] ==
        (df["Age"] >= 60).astype("int8")
    ).all(),

    "Working-age flag": (
        df["Working_Age_Flag"] ==
        df["Age"].between(18, 59).astype("int8")
    ).all()
}

print("Patient feature validation:")

for check, passed in checks.items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

print("\nAge group distribution:")
display(
    df["Age_Group_Standard"]
    .value_counts()
    .sort_index()
)

Patient feature validation:
Age group populated: PASS
Chronic condition flag: PASS
Senior patient flag: PASS
Working-age flag: PASS

Age group distribution:


,count
Age_Group_Standard,
0-17,1496
18-29,5391
30-44,8583
45-59,6881
60-74,4133
75+,3516


### 09.3 - Date Dimension Enhancement

Enhance the date dimension with standardized calendar attributes for time-series analysis, filtering, and period-based reporting in Tableau.

The new fields include year, quarter, month, week, day, and weekend indicators while preserving the original date fields.

In [38]:
# Copy the date dimension before transformation
df = datasets["DIM_DATE"].copy()

# Ensure Date is datetime
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# ---------------------------------------------------------
# Calendar attributes
# ---------------------------------------------------------

df["Year"] = df["Date"].dt.year

df["Quarter"] = "Q" + df["Date"].dt.quarter.astype(str)

df["Quarter_Number"] = df["Date"].dt.quarter

df["Month_Number"] = df["Date"].dt.month

df["Month_Name"] = df["Date"].dt.month_name()

df["Month_Year"] = df["Date"].dt.strftime("%b %Y")

df["Week_Number"] = df["Date"].dt.isocalendar().week.astype("int16")

df["Day_Name"] = df["Date"].dt.day_name()

df["Day_of_Week_Number"] = df["Date"].dt.dayofweek + 1

# Weekend indicator
df["Weekend_Flag"] = (
    df["Date"].dt.dayofweek >= 5
).astype("int8")

# ---------------------------------------------------------
# Store transformed dataset
# ---------------------------------------------------------

datasets["DIM_DATE"] = df

print("Date dimension enhancement completed successfully.")

Date dimension enhancement completed successfully.


In [39]:
# Validate date dimension attributes

df = datasets["DIM_DATE"]

checks = {
    "Date field valid": (
        df["Date"].notna()
    ).all(),

    "Year calculation": (
        df["Year"] == df["Date"].dt.year
    ).all(),

    "Quarter calculation": (
        df["Quarter"] ==
        ("Q" + df["Date"].dt.quarter.astype(str))
    ).all(),

    "Month number calculation": (
        df["Month_Number"] == df["Date"].dt.month
    ).all(),

    "Weekend flag calculation": (
        df["Weekend_Flag"] ==
        (df["Date"].dt.dayofweek >= 5).astype("int8")
    ).all()
}

print("Date dimension validation:")

for check, passed in checks.items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

print("\nDate coverage:")
print(f"Earliest date: {df['Date'].min().date()}")
print(f"Latest date:   {df['Date'].max().date()}")
print(f"Total dates:   {len(df):,}")

Date dimension validation:
Date field valid: PASS
Year calculation: PASS
Quarter calculation: PASS
Month number calculation: PASS
Weekend flag calculation: PASS

Date coverage:
Earliest date: 2023-01-01
Latest date:   2025-12-31
Total dates:   1,096


### 09.4 - Facility & Department Dimension Enhancements

Enhance the facility and department dimensions with standardized analytical attributes for hospital and department-level reporting.

The transformation preserves all original attributes and adds simple classification fields that support consistent Tableau filtering and comparison.

In [40]:
# ---------------------------------------------------------
# FACILITY DIMENSION
# ---------------------------------------------------------

facility = datasets["DIM_FACILITY"].copy()

# Standardize facility type as text
facility["Facility_Type"] = (
    facility["Facility_Type"].astype("string").str.strip()
)

# Create a simple facility identifier label
facility["Facility_Label"] = (
    facility["Facility_ID"].astype("string")
    + " - "
    + facility["Facility_Name"].astype("string")
)

datasets["DIM_FACILITY"] = facility


# ---------------------------------------------------------
# DEPARTMENT DIMENSION
# ---------------------------------------------------------

department = datasets["DIM_DEPARTMENT"].copy()

# Standardize department type/category text fields
for column in department.columns:
    if department[column].dtype == "object":
        department[column] = (
            department[column].astype("string").str.strip()
        )

# Create a standardized department label
department["Department_Label"] = (
    department["Department_ID"].astype("string")
    + " - "
    + department["Department_Name"].astype("string")
)

datasets["DIM_DEPARTMENT"] = department


print("Facility and department feature engineering completed successfully.")

Facility and department feature engineering completed successfully.


In [41]:
# Validate facility and department enhancements

facility = datasets["DIM_FACILITY"]
department = datasets["DIM_DEPARTMENT"]

checks = {
    "Facility labels populated": (
        facility["Facility_Label"].notna()
    ).all(),

    "Facility IDs remain unique": (
        facility["Facility_ID"].is_unique
    ),

    "Department labels populated": (
        department["Department_Label"].notna()
    ).all(),

    "Department IDs remain unique": (
        department["Department_ID"].is_unique
    )
}

print("Facility and department validation:")

for check, passed in checks.items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

print("\nFacility records:", len(facility))
print("Department records:", len(department))

Facility and department validation:
Facility labels populated: PASS
Facility IDs remain unique: PASS
Department labels populated: PASS
Department IDs remain unique: PASS

Facility records: 10
Department records: 15


### 09.5 - Physician, Service, Diagnosis & Payer Enhancements

Standardize the remaining core dimensions and create consistent analytical labels for Tableau reporting.

Original identifiers and descriptive attributes are preserved.

In [43]:
# ---------------------------------------------------------
# PHYSICIAN DIMENSION
# ---------------------------------------------------------

physician = datasets["DIM_PHYSICIAN"].copy()

# Standardize text fields
for column in physician.columns:
    if physician[column].dtype == "object":
        physician[column] = (
            physician[column].astype("string").str.strip()
        )

# Create physician label
physician["Physician_Label"] = (
    physician["Physician_ID"].astype("string")
    + " - "
    + physician["Physician_Name"].astype("string")
)

datasets["DIM_PHYSICIAN"] = physician


# ---------------------------------------------------------
# SERVICE DIMENSION
# ---------------------------------------------------------

service = datasets["DIM_SERVICE"].copy()

for column in service.columns:
    if service[column].dtype == "object":
        service[column] = (
            service[column].astype("string").str.strip()
        )

# Create service label
service["Service_Label"] = (
    service["Service_ID"].astype("string")
    + " - "
    + service["Service_Name"].astype("string")
)

datasets["DIM_SERVICE"] = service


# ---------------------------------------------------------
# DIAGNOSIS DIMENSION
# ---------------------------------------------------------

diagnosis = datasets["DIM_DIAGNOSIS"].copy()

for column in diagnosis.columns:
    if diagnosis[column].dtype == "object":
        diagnosis[column] = (
            diagnosis[column].astype("string").str.strip()
        )

# Create diagnosis label
diagnosis["Diagnosis_Label"] = (
    diagnosis["Diagnosis_ID"].astype("string")
    + " - "
    + diagnosis["Diagnosis_Name"].astype("string")
)

datasets["DIM_DIAGNOSIS"] = diagnosis


# ---------------------------------------------------------
# PAYER DIMENSION
# ---------------------------------------------------------

payer = datasets["DIM_PAYER"].copy()

for column in payer.columns:
    if payer[column].dtype == "object":
        payer[column] = (
            payer[column].astype("string").str.strip()
        )

# Create payer label
payer["Payer_Label"] = (
    payer["Payer_ID"].astype("string")
    + " - "
    + payer["Payer_Name"].astype("string")
)

datasets["DIM_PAYER"] = payer


print("Core dimension feature engineering completed successfully.")

Core dimension feature engineering completed successfully.


In [44]:
# Validate core dimension enhancements

validation = {
    "Physician labels populated": (
        datasets["DIM_PHYSICIAN"]["Physician_Label"].notna().all()
    ),
    "Physician IDs unique": (
        datasets["DIM_PHYSICIAN"]["Physician_ID"].is_unique
    ),
    "Service labels populated": (
        datasets["DIM_SERVICE"]["Service_Label"].notna().all()
    ),
    "Service IDs unique": (
        datasets["DIM_SERVICE"]["Service_ID"].is_unique
    ),
    "Diagnosis labels populated": (
        datasets["DIM_DIAGNOSIS"]["Diagnosis_Label"].notna().all()
    ),
    "Diagnosis IDs unique": (
        datasets["DIM_DIAGNOSIS"]["Diagnosis_ID"].is_unique
    ),
    "Payer labels populated": (
        datasets["DIM_PAYER"]["Payer_Label"].notna().all()
    ),
    "Payer IDs unique": (
        datasets["DIM_PAYER"]["Payer_ID"].is_unique
    )
}

print("Core dimension validation:")

for check, passed in validation.items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

Core dimension validation:
Physician labels populated: PASS
Physician IDs unique: PASS
Service labels populated: PASS
Service IDs unique: PASS
Diagnosis labels populated: PASS
Diagnosis IDs unique: PASS
Payer labels populated: PASS
Payer IDs unique: PASS


### 09.6 - Appointments & Operational Transaction Features

Profile appointment status and timing fields before creating derived operational metrics.

This step identifies the actual appointment categories and validates the available date fields before transformation.

In [45]:
# Inspect appointment status and date fields

df = datasets["APPOINTMENTS"].copy()

print("Appointment columns:")
print(df.columns.tolist())

print("\nAppointment status distribution:")

# Display likely status fields
for column in ["Appointment_Status", "Status", "Appointment_Type"]:
    if column in df.columns:
        print(f"\n{column}:")
        display(df[column].value_counts(dropna=False))

print("\nDate fields:")

for column in df.columns:
    if "Date" in column or "date" in column:
        print(
            f"{column}: "
            f"{pd.to_datetime(df[column], errors='coerce').min()} "
            f"to "
            f"{pd.to_datetime(df[column], errors='coerce').max()}"
        )

Appointment columns:
['Appointment_ID', 'Appointment_Date', 'Patient_ID', 'Facility_ID', 'Department_ID', 'Physician_ID', 'Appointment_Type', 'Appointment_Status', 'Scheduled_Duration_Minutes', 'Booking_Channel', 'Lead_Time_Days']

Appointment status distribution:

Appointment_Status:


,count
Appointment_Status,
Completed,4119
Cancelled,301
No Show,298
Rescheduled,282



Appointment_Type:


,count
Appointment_Type,
New Consultation,1489
Follow-up,1443
Procedure,836
Diagnostic,803
Screening,429



Date fields:
Appointment_Date: 2023-01-01 00:00:00 to 2025-12-31 00:00:00


### 09.6 - Appointments & Operational Transaction Features

Create standardized appointment-status flags and operational classifications to support appointment completion, cancellation, no-show, rescheduling, lead-time, and scheduling analysis.

In [46]:
# Copy the appointments dataset before transformation
df = datasets["APPOINTMENTS"].copy()

# ---------------------------------------------------------
# Standardize text fields
# ---------------------------------------------------------

df["Appointment_Status"] = (
    df["Appointment_Status"]
    .astype("string")
    .str.strip()
)

df["Appointment_Type"] = (
    df["Appointment_Type"]
    .astype("string")
    .str.strip()
)

df["Booking_Channel"] = (
    df["Booking_Channel"]
    .astype("string")
    .str.strip()
)

# Ensure numeric fields are numeric
df["Scheduled_Duration_Minutes"] = pd.to_numeric(
    df["Scheduled_Duration_Minutes"],
    errors="coerce"
)

df["Lead_Time_Days"] = pd.to_numeric(
    df["Lead_Time_Days"],
    errors="coerce"
)

# ---------------------------------------------------------
# Appointment status flags
# ---------------------------------------------------------

df["Completed_Flag"] = (
    df["Appointment_Status"].eq("Completed")
).astype("int8")

df["Cancelled_Flag"] = (
    df["Appointment_Status"].eq("Cancelled")
).astype("int8")

df["No_Show_Flag"] = (
    df["Appointment_Status"].eq("No Show")
).astype("int8")

df["Rescheduled_Flag"] = (
    df["Appointment_Status"].eq("Rescheduled")
).astype("int8")

# ---------------------------------------------------------
# Lead-time classification
# ---------------------------------------------------------

df["Lead_Time_Category"] = pd.cut(
    df["Lead_Time_Days"],
    bins=[-1, 0, 3, 7, 14, np.inf],
    labels=[
        "Same Day",
        "1-3 Days",
        "4-7 Days",
        "8-14 Days",
        "15+ Days"
    ]
)

# ---------------------------------------------------------
# Scheduled duration classification
# ---------------------------------------------------------

df["Duration_Category"] = pd.cut(
    df["Scheduled_Duration_Minutes"],
    bins=[-1, 15, 30, 60, np.inf],
    labels=[
        "≤15 Min",
        "16-30 Min",
        "31-60 Min",
        "60+ Min"
    ]
)

# ---------------------------------------------------------
# Store transformed dataset
# ---------------------------------------------------------

datasets["APPOINTMENTS"] = df

print("Appointment feature engineering completed successfully.")

Appointment feature engineering completed successfully.


In [47]:
# Validate appointment features

df = datasets["APPOINTMENTS"]

# Status flags should be mutually exclusive
status_flag_total = (
    df["Completed_Flag"]
    + df["Cancelled_Flag"]
    + df["No_Show_Flag"]
    + df["Rescheduled_Flag"]
)

checks = {
    "Completed flag": (
        df["Completed_Flag"] ==
        df["Appointment_Status"].eq("Completed").astype("int8")
    ).all(),

    "Cancelled flag": (
        df["Cancelled_Flag"] ==
        df["Appointment_Status"].eq("Cancelled").astype("int8")
    ).all(),

    "No-show flag": (
        df["No_Show_Flag"] ==
        df["Appointment_Status"].eq("No Show").astype("int8")
    ).all(),

    "Rescheduled flag": (
        df["Rescheduled_Flag"] ==
        df["Appointment_Status"].eq("Rescheduled").astype("int8")
    ).all(),

    "Status flags mutually exclusive": (
        status_flag_total == 1
    ).all(),

    "Lead-time category populated": (
        df["Lead_Time_Category"].notna()
    ).all(),

    "Duration category populated": (
        df["Duration_Category"].notna()
    ).all()
}

print("Appointment feature validation:")

for check, passed in checks.items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

print("\nAppointment status totals:")
display(df["Appointment_Status"].value_counts())

print("\nLead-time categories:")
display(df["Lead_Time_Category"].value_counts().sort_index())

Appointment feature validation:
Completed flag: PASS
Cancelled flag: PASS
No-show flag: PASS
Rescheduled flag: PASS
Status flags mutually exclusive: PASS
Lead-time category populated: PASS
Duration category populated: PASS

Appointment status totals:


,count
Appointment_Status,
Completed,4119
Cancelled,301
No Show,298
Rescheduled,282



Lead-time categories:


,count
Lead_Time_Category,
Same Day,122
1-3 Days,336
4-7 Days,435
8-14 Days,781
15+ Days,3326


### 09.7 - Bed Utilization Features

Create analytical classifications for facility bed utilization to support hospital capacity and operational performance analysis.

The original capacity, occupancy, availability, and utilization measures are preserved.

In [48]:
# Copy the bed utilization dataset before transformation
df = datasets["BED_UTILIZATION"].copy()

# Ensure date and numeric fields are correctly typed
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

numeric_columns = [
    "Bed_Capacity",
    "Occupied_Beds",
    "Available_Beds",
    "Bed_Utilization_Pct"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# ---------------------------------------------------------
# Utilization target classification
# ---------------------------------------------------------

df["Utilization_Target_Flag"] = (
    df["Bed_Utilization_Pct"].between(75, 85)
).astype("int8")

# ---------------------------------------------------------
# Utilization performance category
# ---------------------------------------------------------

df["Utilization_Category"] = pd.cut(
    df["Bed_Utilization_Pct"],
    bins=[-np.inf, 74.99, 85, np.inf],
    labels=[
        "Below Target",
        "Within Target",
        "Above Target"
    ]
)

# ---------------------------------------------------------
# Occupancy classification
# ---------------------------------------------------------

df["Capacity_Status"] = np.select(
    [
        df["Bed_Utilization_Pct"] < 75,
        df["Bed_Utilization_Pct"].between(75, 85),
        df["Bed_Utilization_Pct"] > 85
    ],
    [
        "Under Capacity",
        "Optimal Capacity",
        "High Capacity"
    ],
    default="Unknown"
)

# ---------------------------------------------------------
# Available-bed flag
# ---------------------------------------------------------

df["Beds_Available_Flag"] = (
    df["Available_Beds"] > 0
).astype("int8")

# ---------------------------------------------------------
# Store transformed dataset
# ---------------------------------------------------------

datasets["BED_UTILIZATION"] = df

print("Bed utilization feature engineering completed successfully.")

Bed utilization feature engineering completed successfully.


In [49]:
# Validate bed utilization features

df = datasets["BED_UTILIZATION"]

checks = {
    "Utilization target flag": (
        df["Utilization_Target_Flag"] ==
        df["Bed_Utilization_Pct"].between(75, 85).astype("int8")
    ).all(),

    "Utilization category populated": (
        df["Utilization_Category"].notna()
    ).all(),

    "Capacity status populated": (
        df["Capacity_Status"].notna()
    ).all(),

    "Available-bed flag": (
        df["Beds_Available_Flag"] ==
        (df["Available_Beds"] > 0).astype("int8")
    ).all(),

    "Utilization within valid range": (
        df["Bed_Utilization_Pct"].between(0, 100)
    ).all()
}

print("Bed utilization feature validation:")

for check, passed in checks.items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

print("\nUtilization categories:")
display(
    df["Utilization_Category"]
    .value_counts()
    .sort_index()
)

Bed utilization feature validation:
Utilization target flag: PASS
Utilization category populated: PASS
Capacity status populated: PASS
Available-bed flag: PASS
Utilization within valid range: PASS

Utilization categories:


,count
Utilization_Category,
Below Target,4102
Within Target,4499
Above Target,2359


### 09.8 - Claims & Revenue Cycle Features

Create standardized financial and revenue-cycle metrics from the claims dataset.

The derived fields support claim approval, payment, collection, outstanding balance, and payment-speed analysis while preserving the original claim amounts and anomaly flags.

In [50]:
# Copy the claims dataset before transformation
df = datasets["CLAIMS"].copy()

# ---------------------------------------------------------
# Ensure financial and timing fields are numeric
# ---------------------------------------------------------

numeric_columns = [
    "Submitted_Amount_SAR",
    "Approved_Amount_SAR",
    "Paid_Amount_SAR",
    "Days_to_Payment",
    "Outstanding_Amount_SAR"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# ---------------------------------------------------------
# Claim approval rate
# ---------------------------------------------------------

df["Claim_Approval_Rate_Pct"] = np.where(
    df["Submitted_Amount_SAR"] > 0,
    (
        df["Approved_Amount_SAR"] /
        df["Submitted_Amount_SAR"] * 100
    ),
    0
).round(2)

# ---------------------------------------------------------
# Claim payment rate
# ---------------------------------------------------------

df["Claim_Payment_Rate_Pct"] = np.where(
    df["Submitted_Amount_SAR"] > 0,
    (
        df["Paid_Amount_SAR"] /
        df["Submitted_Amount_SAR"] * 100
    ),
    0
).round(2)

# ---------------------------------------------------------
# Outstanding claim flag
# ---------------------------------------------------------

df["Outstanding_Claim_Flag"] = (
    df["Outstanding_Amount_SAR"] > 0
).astype("int8")

# ---------------------------------------------------------
# Payment speed classification
# ---------------------------------------------------------

df["Payment_Speed_Category"] = pd.cut(
    df["Days_to_Payment"],
    bins=[-1, 7, 30, 60, 90, np.inf],
    labels=[
        "0-7 Days",
        "8-30 Days",
        "31-60 Days",
        "61-90 Days",
        "90+ Days"
    ]
)

# ---------------------------------------------------------
# Store transformed dataset
# ---------------------------------------------------------

datasets["CLAIMS"] = df

print("Claims and revenue cycle feature engineering completed successfully.")

Claims and revenue cycle feature engineering completed successfully.


In [51]:
# Validate claims revenue-cycle features

df = datasets["CLAIMS"]

checks = {
    "Approval rate calculation": (
        df["Claim_Approval_Rate_Pct"] ==
        np.where(
            df["Submitted_Amount_SAR"] > 0,
            (
                df["Approved_Amount_SAR"] /
                df["Submitted_Amount_SAR"] * 100
            ),
            0
        ).round(2)
    ).all(),

    "Payment rate calculation": (
        df["Claim_Payment_Rate_Pct"] ==
        np.where(
            df["Submitted_Amount_SAR"] > 0,
            (
                df["Paid_Amount_SAR"] /
                df["Submitted_Amount_SAR"] * 100
            ),
            0
        ).round(2)
    ).all(),

    "Outstanding claim flag": (
        df["Outstanding_Claim_Flag"] ==
        (df["Outstanding_Amount_SAR"] > 0).astype("int8")
    ).all(),

    "Payment speed category populated": (
        df["Payment_Speed_Category"].notna()
    ).all()
}

print("Claims feature validation:")

for check, passed in checks.items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

print("\nPayment speed categories:")
display(
    df["Payment_Speed_Category"]
    .value_counts()
    .sort_index()
)

Claims feature validation:
Approval rate calculation: PASS
Payment rate calculation: PASS
Outstanding claim flag: PASS
Payment speed category populated: PASS

Payment speed categories:


,count
Payment_Speed_Category,
0-7 Days,672
8-30 Days,2508
31-60 Days,3240
61-90 Days,1580
90+ Days,0


### 09.9 - Patient Survey & Experience Features

Create standardized patient-experience metrics from survey responses to support satisfaction, recommendation, service-quality, and negative-feedback analysis in Tableau.

Original survey ratings and feedback categories are preserved.

In [52]:
# Copy the patient survey dataset before transformation
df = datasets["PATIENT_SURVEYS"].copy()

# ---------------------------------------------------------
# Ensure rating fields are numeric
# ---------------------------------------------------------

rating_columns = [
    "Overall_Satisfaction",
    "Staff_Rating",
    "Facility_Rating",
    "Wait_Time_Rating",
    "Cleanliness_Rating"
]

for column in rating_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

# ---------------------------------------------------------
# Recommendation flag
# ---------------------------------------------------------

df["Recommendation_Flag"] = (
    df["Would_Recommend"]
    .astype("string")
    .str.strip()
    .map({"Yes": 1, "No": 0})
    .astype("int8")
)

# ---------------------------------------------------------
# High satisfaction flag
# ---------------------------------------------------------

df["High_Satisfaction_Flag"] = (
    df["Overall_Satisfaction"] >= 4.5
).astype("int8")

# ---------------------------------------------------------
# Average experience rating
# ---------------------------------------------------------

df["Average_Experience_Rating"] = (
    df[rating_columns]
    .mean(axis=1)
    .round(2)
)

# ---------------------------------------------------------
# Negative feedback flag
# ---------------------------------------------------------

df["Negative_Feedback_Flag"] = (
    df["Feedback_Category"]
    .astype("string")
    .str.strip()
    .str.lower()
    .isin([
        "negative",
        "complaint",
        "dissatisfied"
    ])
).astype("int8")

# ---------------------------------------------------------
# Store transformed dataset
# ---------------------------------------------------------

datasets["PATIENT_SURVEYS"] = df

print("Patient survey feature engineering completed successfully.")

Patient survey feature engineering completed successfully.


In [53]:
# Validate patient survey features

df = datasets["PATIENT_SURVEYS"]

checks = {
    "Recommendation flag": (
        df["Recommendation_Flag"].isin([0, 1])
    ).all(),

    "High satisfaction flag": (
        df["High_Satisfaction_Flag"] ==
        (df["Overall_Satisfaction"] >= 4.5).astype("int8")
    ).all(),

    "Average rating calculation": (
        df["Average_Experience_Rating"] ==
        df[
            [
                "Overall_Satisfaction",
                "Staff_Rating",
                "Facility_Rating",
                "Wait_Time_Rating",
                "Cleanliness_Rating"
            ]
        ].mean(axis=1).round(2)
    ).all(),

    "Negative feedback flag": (
        df["Negative_Feedback_Flag"].isin([0, 1])
    ).all()
}

print("Patient survey feature validation:")

for check, passed in checks.items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

print("\nFeedback categories:")
display(
    df["Feedback_Category"]
    .value_counts()
)

Patient survey feature validation:
Recommendation flag: PASS
High satisfaction flag: PASS
Average rating calculation: PASS
Negative feedback flag: PASS

Feedback categories:


,count
Feedback_Category,
Staff,1560
Clinical Care,1277
Waiting Time,1052
Facilities,891
Appointment,476
Billing,438
Other,306


### 09.10 - Patient Experience Classification

Review the distribution of patient satisfaction scores to establish a consistent rule for identifying lower-satisfaction responses.

The classification will support the Patient Experience dashboard and the Negative Feedback Rate KPI without changing the original survey ratings.

In [54]:
# Inspect overall satisfaction distribution

df = datasets["PATIENT_SURVEYS"].copy()

print("Overall satisfaction distribution:")
display(
    df["Overall_Satisfaction"]
    .value_counts()
    .sort_index()
)

print("\nSatisfaction summary:")
display(
    df["Overall_Satisfaction"]
    .describe()
)

Overall satisfaction distribution:


,count
Overall_Satisfaction,
2.40,2
2.50,4
2.70,7
2.80,13
2.90,19
3.00,24
3.10,47
3.20,50
3.30,74



Satisfaction summary:


,Overall_Satisfaction
count,"6,000.00"
mean,4.41
std,0.53
min,2.40
25%,4.10
50%,4.50
75%,4.90
max,5.00


In [55]:
# Copy the patient survey dataset
df = datasets["PATIENT_SURVEYS"].copy()

# ---------------------------------------------------------
# Define low-satisfaction / negative-feedback rule
# ---------------------------------------------------------
# Ratings of 3.5 or below are classified as low satisfaction.
# This provides a consistent proxy for negative patient feedback.

df["Negative_Feedback_Flag"] = (
    df["Overall_Satisfaction"] <= 3.5
).astype("int8")

# Create an explicit satisfaction classification
df["Satisfaction_Category"] = np.select(
    [
        df["Overall_Satisfaction"] <= 3.5,
        df["Overall_Satisfaction"].between(3.51, 4.49),
        df["Overall_Satisfaction"] >= 4.5
    ],
    [
        "Low Satisfaction",
        "Moderate Satisfaction",
        "High Satisfaction"
    ],
    default="Unknown"
)

# Store transformed dataset
datasets["PATIENT_SURVEYS"] = df

print("Patient experience classification completed successfully.")

Patient experience classification completed successfully.


In [57]:
# Validate patient experience classification

df = datasets["PATIENT_SURVEYS"]

checks = {
    "Negative feedback flag": (
        df["Negative_Feedback_Flag"] ==
        (df["Overall_Satisfaction"] <= 3.5).astype("int8")
    ).all(),

    "Satisfaction category populated": (
        df["Satisfaction_Category"].notna()
    ).all(),

    "Valid satisfaction categories": (
        df["Satisfaction_Category"].isin([
            "Low Satisfaction",
            "Moderate Satisfaction",
            "High Satisfaction"
        ])
    ).all()
}

print("Patient experience classification validation:")

for check, passed in checks.items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

print("\nSatisfaction categories:")
display(
    df["Satisfaction_Category"]
    .value_counts()
)

Patient experience classification validation:
Negative feedback flag: PASS
Satisfaction category populated: PASS
Valid satisfaction categories: PASS

Satisfaction categories:


,count
Satisfaction_Category,
High Satisfaction,3128
Moderate Satisfaction,2412
Low Satisfaction,460


### 09.11 - Pharmacy, Laboratory & Imaging Features

Create standardized analytical metrics for pharmacy, laboratory, and imaging transactions.

The transformations support revenue, cost, profitability, processing-time, and service-performance analysis while preserving the original transaction-level measures.

In [58]:
# =========================================================
# PHARMACY
# =========================================================

pharmacy = datasets["PHARMACY"].copy()

# Ensure numeric fields are numeric
for column in [
    "Quantity",
    "Revenue_SAR",
    "Cost_SAR",
    "Gross_Profit_SAR"
]:
    pharmacy[column] = pd.to_numeric(
        pharmacy[column],
        errors="coerce"
    )

# Calculate pharmacy margin
pharmacy["Profit_Margin_Pct"] = np.where(
    pharmacy["Revenue_SAR"] > 0,
    (
        pharmacy["Gross_Profit_SAR"] /
        pharmacy["Revenue_SAR"] * 100
    ),
    0
).round(2)

# Positive-profit flag
pharmacy["Profitable_Transaction_Flag"] = (
    pharmacy["Gross_Profit_SAR"] > 0
).astype("int8")

datasets["PHARMACY"] = pharmacy


# =========================================================
# LAB TRANSACTIONS
# =========================================================

lab = datasets["LAB_TRANSACTIONS"].copy()

for column in [
    "Processing_Time_Hours",
    "Revenue_SAR",
    "Cost_SAR"
]:
    lab[column] = pd.to_numeric(
        lab[column],
        errors="coerce"
    )

# Laboratory gross profit
lab["Gross_Profit_SAR"] = (
    lab["Revenue_SAR"] -
    lab["Cost_SAR"]
).round(2)

# Laboratory profit margin
lab["Profit_Margin_Pct"] = np.where(
    lab["Revenue_SAR"] > 0,
    (
        lab["Gross_Profit_SAR"] /
        lab["Revenue_SAR"] * 100
    ),
    0
).round(2)

# Processing-time performance flag
lab["Processing_Within_24H_Flag"] = (
    lab["Processing_Time_Hours"] <= 24
).astype("int8")

datasets["LAB_TRANSACTIONS"] = lab


# =========================================================
# IMAGING
# =========================================================

imaging = datasets["IMAGING"].copy()

for column in [
    "Waiting_Time_Minutes",
    "Report_Time_Hours",
    "Revenue_SAR",
    "Cost_SAR"
]:
    imaging[column] = pd.to_numeric(
        imaging[column],
        errors="coerce"
    )

# Imaging gross profit
imaging["Gross_Profit_SAR"] = (
    imaging["Revenue_SAR"] -
    imaging["Cost_SAR"]
).round(2)

# Imaging profit margin
imaging["Profit_Margin_Pct"] = np.where(
    imaging["Revenue_SAR"] > 0,
    (
        imaging["Gross_Profit_SAR"] /
        imaging["Revenue_SAR"] * 100
    ),
    0
).round(2)

# Imaging waiting-time performance flag
imaging["Waiting_Time_Within_Target_Flag"] = (
    imaging["Waiting_Time_Minutes"] <= 30
).astype("int8")

datasets["IMAGING"] = imaging


print("Pharmacy, laboratory, and imaging feature engineering completed successfully.")

Pharmacy, laboratory, and imaging feature engineering completed successfully.


In [59]:
# Validate pharmacy, laboratory, and imaging features

pharmacy = datasets["PHARMACY"]
lab = datasets["LAB_TRANSACTIONS"]
imaging = datasets["IMAGING"]

checks = {
    "Pharmacy margin calculation": (
        pharmacy["Profit_Margin_Pct"] ==
        np.where(
            pharmacy["Revenue_SAR"] > 0,
            (
                pharmacy["Gross_Profit_SAR"] /
                pharmacy["Revenue_SAR"] * 100
            ),
            0
        ).round(2)
    ).all(),

    "Pharmacy profitable flag": (
        pharmacy["Profitable_Transaction_Flag"] ==
        (pharmacy["Gross_Profit_SAR"] > 0).astype("int8")
    ).all(),

    "Lab gross profit calculation": (
        lab["Gross_Profit_SAR"] ==
        (lab["Revenue_SAR"] - lab["Cost_SAR"]).round(2)
    ).all(),

    "Lab margin calculation": (
        lab["Profit_Margin_Pct"] ==
        np.where(
            lab["Revenue_SAR"] > 0,
            (
                lab["Gross_Profit_SAR"] /
                lab["Revenue_SAR"] * 100
            ),
            0
        ).round(2)
    ).all(),

    "Lab processing flag": (
        lab["Processing_Within_24H_Flag"] ==
        (lab["Processing_Time_Hours"] <= 24).astype("int8")
    ).all(),

    "Imaging gross profit calculation": (
        imaging["Gross_Profit_SAR"] ==
        (imaging["Revenue_SAR"] - imaging["Cost_SAR"]).round(2)
    ).all(),

    "Imaging margin calculation": (
        imaging["Profit_Margin_Pct"] ==
        np.where(
            imaging["Revenue_SAR"] > 0,
            (
                imaging["Gross_Profit_SAR"] /
                imaging["Revenue_SAR"] * 100
            ),
            0
        ).round(2)
    ).all(),

    "Imaging waiting-time flag": (
        imaging["Waiting_Time_Within_Target_Flag"] ==
        (imaging["Waiting_Time_Minutes"] <= 30).astype("int8")
    ).all()
}

print("Clinical transaction feature validation:")

for check, passed in checks.items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

Clinical transaction feature validation:
Pharmacy margin calculation: PASS
Pharmacy profitable flag: PASS
Lab gross profit calculation: PASS
Lab margin calculation: PASS
Lab processing flag: PASS
Imaging gross profit calculation: PASS
Imaging margin calculation: PASS
Imaging waiting-time flag: PASS


### 09.12 - Final Feature Audit

Perform a final audit of all engineered fields and dataset row counts before completing the transformation stage.

The audit confirms that derived features were created successfully and that no records were unintentionally added or removed.

In [60]:
# Expected engineered columns by dataset

EXPECTED_FEATURES = {
    "FACT_ENCOUNTER": [
        "Revenue_Per_Treatment_Minute_SAR",
        "Cost_Per_Treatment_Minute_SAR",
        "Total_Service_Time_Minutes",
        "Wait_Time_Within_Target_Flag",
        "Satisfaction_Within_Target_Flag",
        "Inpatient_Flag",
        "Outpatient_Flag"
    ],

    "DIM_PATIENT": [
        "Age_Group_Standard",
        "Senior_Patient_Flag",
        "Working_Age_Flag"
    ],

    "DIM_DATE": [
        "Year",
        "Quarter",
        "Quarter_Number",
        "Month_Number",
        "Month_Name",
        "Month_Year",
        "Week_Number",
        "Day_Name",
        "Day_of_Week_Number",
        "Weekend_Flag"
    ],

    "DIM_FACILITY": [
        "Facility_Label"
    ],

    "DIM_DEPARTMENT": [
        "Department_Label"
    ],

    "DIM_PHYSICIAN": [
        "Physician_Label"
    ],

    "DIM_SERVICE": [
        "Service_Label"
    ],

    "DIM_DIAGNOSIS": [
        "Diagnosis_Label"
    ],

    "DIM_PAYER": [
        "Payer_Label"
    ],

    "APPOINTMENTS": [
        "Completed_Flag",
        "Cancelled_Flag",
        "No_Show_Flag",
        "Rescheduled_Flag",
        "Lead_Time_Category",
        "Duration_Category"
    ],

    "BED_UTILIZATION": [
        "Utilization_Target_Flag",
        "Utilization_Category",
        "Capacity_Status",
        "Beds_Available_Flag"
    ],

    "CLAIMS": [
        "Paid_Above_Approved_Flag",
        "Paid_Above_Submitted_Flag",
        "Claim_Approval_Rate_Pct",
        "Claim_Payment_Rate_Pct",
        "Outstanding_Claim_Flag",
        "Payment_Speed_Category"
    ],

    "PATIENT_SURVEYS": [
        "Recommendation_Flag",
        "High_Satisfaction_Flag",
        "Average_Experience_Rating",
        "Negative_Feedback_Flag",
        "Satisfaction_Category"
    ],

    "PHARMACY": [
        "Profit_Margin_Pct",
        "Profitable_Transaction_Flag"
    ],

    "LAB_TRANSACTIONS": [
        "Gross_Profit_SAR",
        "Profit_Margin_Pct",
        "Processing_Within_24H_Flag"
    ],

    "IMAGING": [
        "Gross_Profit_SAR",
        "Profit_Margin_Pct",
        "Waiting_Time_Within_Target_Flag"
    ]
}

# Original expected row counts
EXPECTED_ROWS = {
    "FACT_ENCOUNTER": 75000,
    "DIM_PATIENT": 30000,
    "DIM_DATE": 1096,
    "DIM_FACILITY": 10,
    "DIM_DEPARTMENT": 15,
    "DIM_PHYSICIAN": 60,
    "DIM_SERVICE": 25,
    "DIM_DIAGNOSIS": 30,
    "DIM_PAYER": 8,
    "APPOINTMENTS": 5000,
    "BED_UTILIZATION": 10960,
    "CLAIMS": 8000,
    "PATIENT_SURVEYS": 6000,
    "PHARMACY": 7000,
    "LAB_TRANSACTIONS": 7000,
    "IMAGING": 5000
}

# ---------------------------------------------------------
# Run feature and row-count audit
# ---------------------------------------------------------

audit_results = []

for sheet in WORKING_SHEETS:
    df = datasets[sheet]

    expected_features = EXPECTED_FEATURES.get(sheet, [])

    missing_features = [
        column
        for column in expected_features
        if column not in df.columns
    ]

    expected_rows = EXPECTED_ROWS.get(sheet)
    actual_rows = len(df)

    audit_results.append({
        "Dataset": sheet,
        "Expected Rows": expected_rows,
        "Actual Rows": actual_rows,
        "Row Count Passed": actual_rows == expected_rows,
        "Missing Features": ", ".join(missing_features)
        if missing_features else "None",
        "Feature Check Passed": len(missing_features) == 0
    })

feature_audit = pd.DataFrame(audit_results)

print("Final feature audit completed.")
display(feature_audit)

Final feature audit completed.


,Dataset,Expected Rows,Actual Rows,Row Count Passed,Missing Features,Feature Check Passed
0,FACT_ENCOUNTER,75000,75000,True,None,True
1,DIM_PATIENT,30000,30000,True,None,True
2,DIM_DATE,1096,1096,True,None,True
3,DIM_FACILITY,10,10,True,None,True
4,DIM_DEPARTMENT,15,15,True,None,True
5,DIM_PHYSICIAN,60,60,True,None,True
6,DIM_SERVICE,25,25,True,None,True
7,DIM_DIAGNOSIS,30,30,True,None,True
8,DIM_PAYER,8,8,True,None,True
9,APPOINTMENTS,5000,5000,True,None,True


In [61]:
# Final audit summary

row_check_passed = feature_audit["Row Count Passed"].all()
feature_check_passed = feature_audit["Feature Check Passed"].all()

print("FINAL FEATURE AUDIT")
print("-------------------")
print(
    f"Datasets audited: {len(feature_audit)}"
)
print(
    f"Row-count checks passed: "
    f"{feature_audit['Row Count Passed'].sum()} / "
    f"{len(feature_audit)}"
)
print(
    f"Feature checks passed: "
    f"{feature_audit['Feature Check Passed'].sum()} / "
    f"{len(feature_audit)}"
)

print("\nOverall result:")

if row_check_passed and feature_check_passed:
    print("PASS — All engineered features exist and row counts are unchanged.")
else:
    print("REVIEW — One or more audit checks require attention.")

FINAL FEATURE AUDIT
-------------------
Datasets audited: 16
Row-count checks passed: 16 / 16
Feature checks passed: 16 / 16

Overall result:
PASS — All engineered features exist and row counts are unchanged.


# 10 - Final Dataset Preparation

Prepare the transformed datasets for final Excel export.

This stage standardizes column organization and confirms that all analytical fields are retained without changing underlying values or row counts.

In [62]:
# Standardize column order across all transformed datasets
# Original source columns remain first; engineered columns follow.

for sheet in WORKING_SHEETS:
    df = datasets[sheet]

    # Identify columns that existed before feature engineering
    original_columns = [
        column for column in df.columns
        if column not in EXPECTED_FEATURES.get(sheet, [])
    ]

    # Add engineered columns in their current order
    engineered_columns = [
        column for column in EXPECTED_FEATURES.get(sheet, [])
        if column in df.columns
    ]

    # Reorder dataset
    datasets[sheet] = df[
        original_columns + engineered_columns
    ].copy()

print("Final column ordering completed successfully.")

Final column ordering completed successfully.


In [63]:
# Validate final column ordering and dataset integrity

final_checks = []

for sheet in WORKING_SHEETS:
    df = datasets[sheet]

    # Check expected row count
    row_check = len(df) == EXPECTED_ROWS[sheet]

    # Check all engineered columns exist
    missing_features = [
        column
        for column in EXPECTED_FEATURES.get(sheet, [])
        if column not in df.columns
    ]

    feature_check = len(missing_features) == 0

    # Check for duplicate column names
    duplicate_columns = df.columns[df.columns.duplicated()].tolist()

    column_check = len(duplicate_columns) == 0

    final_checks.append({
        "Dataset": sheet,
        "Rows Valid": row_check,
        "Features Valid": feature_check,
        "Duplicate Columns": len(duplicate_columns),
        "Passed": (
            row_check and
            feature_check and
            column_check
        )
    })

final_preparation_audit = pd.DataFrame(final_checks)

print("Final dataset preparation validation:")
display(final_preparation_audit)

print("\nOverall result:")

if final_preparation_audit["Passed"].all():
    print("PASS — All datasets are ready for final workbook export.")
else:
    print("REVIEW — One or more datasets require attention.")

Final dataset preparation validation:


,Dataset,Rows Valid,Features Valid,Duplicate Columns,Passed
0,FACT_ENCOUNTER,True,True,0,True
1,DIM_PATIENT,True,True,0,True
2,DIM_DATE,True,True,0,True
3,DIM_FACILITY,True,True,0,True
4,DIM_DEPARTMENT,True,True,0,True
5,DIM_PHYSICIAN,True,True,0,True
6,DIM_SERVICE,True,True,0,True
7,DIM_DIAGNOSIS,True,True,0,True
8,DIM_PAYER,True,True,0,True
9,APPOINTMENTS,True,True,0,True



Overall result:
PASS — All datasets are ready for final workbook export.


## 10.2 - Final Workbook Architecture Review

Confirm the final workbook structure before export.

The workbook will retain:
- 16 transformed analytical datasets
- MODEL_GUIDE for data model and relationship documentation
- DATA_DICTIONARY for field definitions and business context

All analytical datasets will retain their original grain and will be used in Tableau through a multi-fact dimensional model.

In [64]:
# Define the final workbook structure
# Informational sheets are retained separately from analytical datasets.

FINAL_SHEETS = WORKING_SHEETS + INFO_SHEETS

print("Final workbook architecture:")
print(f"Analytical datasets: {len(WORKING_SHEETS)}")
print(f"Documentation sheets: {len(INFO_SHEETS)}")
print(f"Total sheets: {len(FINAL_SHEETS)}")

print("\nAnalytical datasets:")
for i, sheet in enumerate(WORKING_SHEETS, start=1):
    print(f"{i:02d}. {sheet}")

print("\nDocumentation sheets:")
for i, sheet in enumerate(INFO_SHEETS, start=1):
    print(f"{i:02d}. {sheet}")

Final workbook architecture:
Analytical datasets: 16
Documentation sheets: 2
Total sheets: 18

Analytical datasets:
01. FACT_ENCOUNTER
02. DIM_PATIENT
03. DIM_DATE
04. DIM_FACILITY
05. DIM_DEPARTMENT
06. DIM_PHYSICIAN
07. DIM_SERVICE
08. DIM_DIAGNOSIS
09. DIM_PAYER
10. APPOINTMENTS
11. BED_UTILIZATION
12. CLAIMS
13. PATIENT_SURVEYS
14. PHARMACY
15. LAB_TRANSACTIONS
16. IMAGING

Documentation sheets:
01. MODEL_GUIDE
02. DATA_DICTIONARY


In [65]:
# Validate that all expected sheets are present in the final architecture

architecture_check = (
    set(FINAL_SHEETS) == set(sheet_names)
    and len(FINAL_SHEETS) == 18
    and len(WORKING_SHEETS) == 16
    and len(INFO_SHEETS) == 2
)

if architecture_check:
    print("PASS — Final workbook architecture is complete and consistent.")
else:
    print("REVIEW — Workbook architecture does not match expectations.")

PASS — Final workbook architecture is complete and consistent.


## 10.3 - Final Workbook Export

Export the validated transformed datasets into a single Excel workbook.

The final workbook retains all analytical datasets and supporting documentation required for Tableau development and project handover.

In [66]:
# Define the final output workbook name
FINAL_FILE = "Healthcare_Operations_Patient_Performance_Analytics_Final.xlsx"

# Export all transformed datasets and documentation sheets
with pd.ExcelWriter(FINAL_FILE, engine="openpyxl") as writer:

    # Write transformed analytical datasets
    for sheet in WORKING_SHEETS:
        datasets[sheet].to_excel(
            writer,
            sheet_name=sheet,
            index=False
        )

    # Preserve the original documentation sheets
    for sheet in INFO_SHEETS:
        documentation_df = pd.read_excel(
            excel,
            sheet_name=sheet
        )

        documentation_df.to_excel(
            writer,
            sheet_name=sheet,
            index=False
        )

print(f"Final workbook exported successfully: {FINAL_FILE}")

Final workbook exported successfully: Healthcare_Operations_Patient_Performance_Analytics_Final.xlsx


In [67]:
# Re-open the exported workbook to validate the final file
final_excel = pd.ExcelFile(FINAL_FILE)

print("Export validation:")
print(f"File exists: {os.path.exists(FINAL_FILE)}")
print(f"Total sheets: {len(final_excel.sheet_names)}")
print(f"Expected sheets: {len(FINAL_SHEETS)}")

# Validate sheet names and row counts
export_checks = []

for sheet in WORKING_SHEETS:
    exported_df = pd.read_excel(final_excel, sheet_name=sheet)

    export_checks.append({
        "Dataset": sheet,
        "Rows": len(exported_df),
        "Expected Rows": EXPECTED_ROWS[sheet],
        "Row Count Valid": len(exported_df) == EXPECTED_ROWS[sheet],
        "Columns": len(exported_df.columns)
    })

export_audit = pd.DataFrame(export_checks)

display(export_audit)

if (
    final_excel.sheet_names == FINAL_SHEETS
    and export_audit["Row Count Valid"].all()
):
    print("\nPASS — Final workbook exported and validated successfully.")
else:
    print("\nREVIEW — Export validation requires attention.")

Export validation:
File exists: True
Total sheets: 18
Expected sheets: 18


,Dataset,Rows,Expected Rows,Row Count Valid,Columns
0,FACT_ENCOUNTER,75000,75000,True,33
1,DIM_PATIENT,30000,30000,True,13
2,DIM_DATE,1096,1096,True,13
3,DIM_FACILITY,10,10,True,9
4,DIM_DEPARTMENT,15,15,True,7
5,DIM_PHYSICIAN,60,60,True,8
6,DIM_SERVICE,25,25,True,7
7,DIM_DIAGNOSIS,30,30,True,4
8,DIM_PAYER,8,8,True,6
9,APPOINTMENTS,5000,5000,True,17



PASS — Final workbook exported and validated successfully.


In [68]:
# Download the validated final workbook to your computer
from google.colab import files

files.download(FINAL_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>